# Linear and Logistic Regression (and their Regularized Variants)

This notebook provides a university-level introduction to **Linear Regression** (for continuous targets) and **Logistic Regression** (for binary / multi-class targets), together with their regularized variants (Ridge, Lasso, Elastic Net, L1/L2-penalized logistic regression). For each model we present both the `scikit-learn` implementation (for prediction / pipelines) and the `statsmodels` implementation (for classical statistical inference).

Author: Michał Woźniak

## Technical Setup

This notebook is part of the course repository, which already ships a complete, locked environment (`pyproject.toml` + `uv.lock`). You do **not** need to install packages one by one.

```bash
# from the root of the repository
uv sync
uv run jupyter lab
```

A detailed, step-by-step installation guide (what `uv` is, how to install it on Windows / macOS / Linux, how to select the kernel in VS Code, troubleshooting) is available in [`SETUP.md`](../SETUP.md).

**Packages used in this notebook:** `numpy`, `pandas`, `matplotlib`, `seaborn`, `scikit-learn`, `statsmodels`.

**Data:** California Housing (`fetch_california_housing`) is downloaded on first use and then cached locally in `~/scikit_learn_data`, so the first run needs internet access.

# 1. Introduction to Linear and Logistic Regression

## What are Linear and Logistic Regression?

Both are **supervised**, **parametric**, **eager-learning** algorithms that model the target as a function of a **linear combination** of the input features. They differ only in what that linear combination is used for:

| | Linear Regression | Logistic Regression |
|---|---|---|
| **Target** | Continuous (`y ∈ ℝ`) | Binary / categorical (`y ∈ {0,1}` or `{1, …, K}`) |
| **Prediction** | $\hat{y} = x^\top w + b$ | $\hat{p} = \sigma(x^\top w + b)$ |
| **Loss** | Mean Squared Error (MSE) | Binary / categorical cross-entropy |
| **Solver** | Closed-form (normal equation) | Iterative (no closed form) |
| **Probabilistic story** | MLE under Gaussian noise | MLE under Bernoulli / Multinoulli response |

### Key Characteristics

1. **Parametric**: The model is fully described by a fixed number of parameters $w$ (coefficients) and $b$ (intercept). Training data is summarized into these parameters and then discarded.
2. **Interpretable**: Each coefficient $w_j$ has a direct interpretation.
3. **Fast training and prediction**: Training is $O(np^2 + p^3)$ for OLS; logistic regression is iterative but also fast.
4. **Strong baseline**: Any non-linear model should beat these to justify its complexity.

### Mathematical Foundation — Linear Regression

#### Ordinary Least Squares (OLS)

Assume $y_i = x_i^\top w + b + \varepsilon_i$ with $\mathbb{E}[\varepsilon_i] = 0$. OLS chooses $w$ to minimize the sum of squared residuals $\|y - Xw\|_2^2$. Dividing by $2n$ does not change the minimizer and gives the (half) **mean** squared error that we will use as the loss throughout:

$$\mathcal{L}(w) = \frac{1}{2n}\sum_{i=1}^{n}(y_i - x_i^\top w)^2 = \frac{1}{2n}\|y - Xw\|_2^2$$

Setting $\nabla_w \mathcal{L}(w) = 0$ gives the **normal equation**:

$$X^\top X \, w = X^\top y \quad\Longrightarrow\quad \hat{w} = (X^\top X)^{-1} X^\top y$$

provided $X^\top X$ is invertible (i.e. the columns of $X$ are linearly independent).

#### Geometric Interpretation
$X\hat{w}$ is the **orthogonal projection** of $y$ onto the column space of $X$. The residual vector $y - X\hat{w}$ is orthogonal to every feature column.

#### Probabilistic Interpretation (MLE)
If $y_i\mid x_i \sim \mathcal{N}(x_i^\top w,\sigma^2)$ then maximizing the log-likelihood is equivalent to minimizing the sum of squared errors — so **OLS = MLE under Gaussian noise**.

#### Gauss–Markov Theorem
Under linearity, exogeneity ($\mathbb{E}[\varepsilon\mid X]=0$), homoskedasticity and no perfect multicollinearity, OLS is the **Best Linear Unbiased Estimator (BLUE)**: smallest variance among linear unbiased estimators.

---

### Mathematical Foundation — Logistic Regression

#### The Sigmoid and the Log-Odds

For binary classification with $y_i \in \{0,1\}$ we use the **logistic (sigmoid) function**:

$$\sigma(z) = \frac{1}{1 + e^{-z}} \in (0,1)$$

The model is

$$\mathbb{P}(y=1\mid x) = \sigma(x^\top w + b)$$

Equivalently, the log-odds ("logit") is linear in $x$:

$$\log\frac{\mathbb{P}(y=1\mid x)}{\mathbb{P}(y=0\mid x)} = x^\top w + b$$

So a coefficient $w_j$ means: “a one-unit increase in $x_j$ (holding others fixed) increases the log-odds of $y=1$ by $w_j$”, or equivalently multiplies the odds by $e^{w_j}$.

#### Cross-Entropy Loss

The negative log-likelihood under a Bernoulli model gives:

$$\mathcal{L}(w,b) = -\frac{1}{n}\sum_{i=1}^n \Bigl[y_i \log \hat{p}_i + (1-y_i)\log(1-\hat{p}_i)\Bigr]$$

where $\hat{p}_i = \sigma(x_i^\top w + b)$. The gradient is remarkably clean:

$$\nabla_w \mathcal{L} = \frac{1}{n}\sum_i (\hat{p}_i - y_i)\, x_i = \frac{1}{n} X^\top (\hat{p} - y)$$
$$\nabla_b \mathcal{L} = \frac{1}{n}\sum_i (\hat{p}_i - y_i)$$

#### Why No Closed Form?
The cross-entropy loss is **convex** (so every local minimum is a global one) but involves $\sigma(\cdot)$, so there is no closed-form solution. One caveat: a *finite* minimizer exists only when the classes are **not perfectly separable** by a hyperplane — otherwise the loss can be pushed towards zero by scaling $w$ up without bound, and the MLE does not exist (see the note on separation in Section 8). In practice we solve it iteratively with:
- **Gradient descent / SGD** (what we’ll implement from scratch)
- **Newton’s method** → known as **Iteratively Reweighted Least Squares (IRLS)** in this context, and what `statsmodels` and old-school GLM packages use
- **L-BFGS / liblinear / SAGA** — the defaults in `sklearn`

#### Multiclass Extension: Softmax Regression
For $K$ classes we generalize to

$$\mathbb{P}(y=k\mid x) = \frac{e^{x^\top w_k + b_k}}{\sum_{j=1}^K e^{x^\top w_j + b_j}}$$

This is also called **multinomial logistic regression** or **softmax regression**.

---

### Why Regularization?

Both models suffer from the same two failure modes:

1. **Multicollinearity**: correlated features lead to unstable coefficients with huge variance.
2. **High-dimensional or small-sample problems**: when $p \gtrsim n$, the unregularized problem is ill-posed and severely overfits.

**Regularization** fixes this by adding a penalty on the size of $w$, trading a little bias for a lot less variance.

| Variant | Penalty added to the loss $\tfrac{1}{2n}\|y - Xw\|_2^2$ | Effect |
|---|---|---|
| **Ridge (L2)** | $\tfrac{\lambda}{2}\|w\|_2^2$ | smooth shrinkage; stabilizes correlated features |
| **Lasso (L1)** | $\lambda \|w\|_1$ | drives coefficients exactly to zero → variable selection |
| **Elastic Net** | $\lambda\rho\|w\|_1 + \tfrac{\lambda(1-\rho)}{2}\|w\|_2^2$ | both at once |

Regularized logistic regression works identically — just add the penalty term to the cross-entropy loss.

**How `sklearn` scales the penalty (read the docstrings, not your memory!)** — every estimator has its own convention, and mixing them up changes the amount of shrinkage by a factor of $n$:

| Estimator | Objective actually minimized | Relation to $\lambda$ above |
|---|---|---|
| `Ridge(alpha)` | $\|y - Xw\|_2^2 + \alpha\|w\|_2^2$ | $\alpha = n\lambda$ |
| `Lasso(alpha)` | $\tfrac{1}{2n}\|y - Xw\|_2^2 + \alpha\|w\|_1$ | $\alpha = \lambda$ |
| `ElasticNet(alpha, l1_ratio=ρ)` | $\tfrac{1}{2n}\|y - Xw\|_2^2 + \alpha\rho\|w\|_1 + \tfrac{\alpha(1-\rho)}{2}\|w\|_2^2$ | $\alpha = \lambda$ |
| `LogisticRegression(C, l1_ratio=ρ)` | $C\sum_{i=1}^n \ell_i(w,b) + \rho\|w\|_1 + \tfrac{1-\rho}{2}\|w\|_2^2$ | $C = \tfrac{1}{n\lambda}$ |

(here $\ell_i$ is the per-observation cross-entropy, so the logistic objective uses the **sum**, not the mean, of the losses). Two consequences worth remembering:

* `Ridge(alpha=1)` on $n = 3{,}500$ rows is a *much* weaker penalty than `Lasso(alpha=1)`; to compare like with like use `Ridge(alpha=n * a)` against `Lasso(alpha=a)`.
* For logistic regression a larger `C` means *less* regularization (SVM convention), and $C = 1/(n\lambda)$ — **not** $1/\lambda$ — in the mean-loss convention of this notebook.


## Key Hyperparameters

### 1. Regularization Strength (`alpha` for the linear models, `C` for logistic)
- Larger `alpha` = stronger penalty; larger `C` = **weaker** penalty. With the notebook's mean-loss convention, `alpha` = $n\lambda$ (Ridge) or $\lambda$ (Lasso / Elastic Net) and `C` = $1/(n\lambda)$ — see the table above.
- **`alpha = 0` / `C = np.inf`**: pure OLS / unpenalized logistic — minimum bias, maximum variance.
- **Large $\alpha$ / small $C$**: heavy shrinkage — coefficients close to zero, high bias but low variance.
- **Best practice**: always search on a **logarithmic scale** (e.g. $10^{-4}$ to $10^{4}$).

### 2. L1 Ratio ($\rho$, Elastic Net)
- $\rho = 0$: pure L2 (Ridge). $\rho = 1$: pure L1 (Lasso). Mixture in between.
- Start with $\rho = 0.5$ and tune with CV.

### 3. `fit_intercept`
- Almost always keep it `True`. Setting it `False` forces the line through the origin — rarely what you want.

### 4. `class_weight` (Logistic only)
- Use `class_weight='balanced'` for imbalanced data.

## Advantages and Limitations

### Advantages ✓
1. **Simple and interpretable** — each coefficient has a direct meaning.
2. **Convex optimization** — closed-form global optimum for OLS; logistic is iterative but convex, so no bad local minima (a finite optimum requires non-separable classes).
3. **Statistical inference** — `statsmodels` gives p-values, CIs, hypothesis tests.
4. **Fast** — training and prediction scale very well with $n$ and $p$.
5. **Excellent baseline** — any non-linear model should beat these to justify its complexity.

### Limitations ✗
1. **Linearity assumption** — miss non-linear relationships unless you feature-engineer them.
2. **Sensitive to outliers** — squared loss heavily penalizes large residuals; logistic is milder but still pulled by extremes.
3. **Multicollinearity blows up variance** — unless regularized.
4. **Require feature scaling for regularized variants** — the L1/L2 penalty treats every coefficient the same regardless of the feature scale.
5. **Decision boundary is a hyperplane (logistic)** — no curvature without feature engineering.

## Critical Preprocessing: Feature Scaling

**For plain OLS**, scaling is *not* mathematically required — the solution is invariant to linear rescaling of individual features.

**For Ridge / Lasso / Elastic Net** (both linear and logistic), scaling is **essential**. Consider two features:
- Feature 1: Age (range 20–80)
- Feature 2: Income (range 20,000–200,000)

The penalty $\alpha\|w\|_2^2$ penalizes `w_income` much less than `w_age` because a small `w_income` already produces a large contribution to the prediction. After standardization the penalty applies fairly to all features.

**Common scaling methods:**

1. **Standardization (Z-score normalization)**: zero mean, unit variance
   $$x' = \frac{x - \mu}{\sigma}$$
2. **Min-Max Scaling (Normalization)**: to $[0,1]$
   $$x' = \frac{x - x_{\min}}{x_{\max} - x_{\min}}$$
3. **Robust Scaling**: median and IQR (outlier-safe)
   $$x' = \frac{x - \text{median}}{\text{IQR}}$$

**Best practice**: always fit the scaler on the training set only, then transform both train and test.

# 2. Linear Regression: From-Scratch Implementation

In this section we implement linear regression from scratch to understand the mechanics. We build:
1. A **closed-form OLS** estimator using the normal equation.
2. A **gradient-descent OLS** estimator that optimizes the MSE iteratively.

Then we compare their behavior on the California Housing dataset.

In [ ]:
import warnings

from sklearn.exceptions import ConvergenceWarning

warnings.filterwarnings("ignore", category=ConvergenceWarning)

# Import necessary libraries
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

# Set random seed for reproducibility
np.random.seed(42)

# Matplotlib configuration for better plots
plt.style.use("seaborn-v0_8-darkgrid")
sns.set_palette("husl")

print("Libraries imported successfully!")

In [ ]:
# Load the California Housing dataset (our regression benchmark)
from sklearn.datasets import fetch_california_housing

housing = fetch_california_housing()
# Subsample to keep from-scratch experiments snappy. Sample WITHOUT replacement:
# duplicated rows would later end up in both the training and the test split (leakage).
rng = np.random.default_rng(42)
subsample_idx = rng.choice(housing.data.shape[0], size=5000, replace=False)
X_reg, y_reg = housing.data[subsample_idx], housing.target[subsample_idx]

print("California Housing Dataset Loaded")
print(f"Features shape: {X_reg.shape}")
print(f"Target shape:   {y_reg.shape}")
print(f"\nFeature names: {housing.feature_names}")
print("Target: Median house value (in $100,000s)")
print(f"\nFirst 5 samples:\n{X_reg[:5]}")
print(f"First 5 targets: {y_reg[:5]}")

## 2.1 Closed-Form OLS (Normal Equation)

Recall the normal equation:
$$\hat{w} = (X^\top X)^{-1} X^\top y$$

In practice we never actually invert $X^\top X$ — we solve the linear system with `np.linalg.solve` for numerical stability.

In [ ]:
class OLSRegressorClosedForm:
    """
    Ordinary Least Squares regression using the normal equation.

    Solves (X^T X) w = X^T y directly with np.linalg.solve
    (more numerically stable than computing the inverse).
    """

    def __init__(self, fit_intercept=True):
        self.fit_intercept = fit_intercept

    def _add_intercept(self, X):
        if self.fit_intercept:
            return np.hstack([np.ones((X.shape[0], 1)), X])
        return X

    def fit(self, X_train, y_train):
        X_train = np.asarray(X_train, dtype=float)
        y_train = np.asarray(y_train, dtype=float)

        X_ = self._add_intercept(X_train)
        XtX = X_.T @ X_
        Xty = X_.T @ y_train
        theta = np.linalg.solve(XtX, Xty)

        if self.fit_intercept:
            self.intercept_ = theta[0]
            self.coef_ = theta[1:]
        else:
            self.intercept_ = 0.0
            self.coef_ = theta
        return self

    def predict(self, X):
        X = np.asarray(X, dtype=float)
        return X @ self.coef_ + self.intercept_

    def score(self, X, y):
        """Return the coefficient of determination R^2."""
        y = np.asarray(y, dtype=float)
        y_pred = self.predict(X)
        ss_res = np.sum((y - y_pred) ** 2)
        ss_tot = np.sum((y - y.mean()) ** 2)
        return 1.0 - ss_res / ss_tot


print("OLSRegressorClosedForm class created successfully!")

### Split data and (optionally) scale features

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(X_reg, y_reg, test_size=0.3, random_state=42)

print(f"Training set size: {X_train_r.shape[0]}")
print(f"Test set size:     {X_test_r.shape[0]}")

# For plain OLS, scaling is not strictly required, but it makes
# coefficient magnitudes comparable and is REQUIRED for the
# regularized variants later.
scaler_r = StandardScaler()
scaler_r.fit(X_train_r)
X_train_rs = scaler_r.transform(X_train_r)
X_test_rs = scaler_r.transform(X_test_r)

print("\nBefore scaling (first 3 training samples):")
print(X_train_r[:3])
print("\nAfter scaling (first 3 training samples):")
print(X_train_rs[:3])

### Train the closed-form OLS model

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

ols_cf = OLSRegressorClosedForm(fit_intercept=True)

start_time = time.time()
ols_cf.fit(X_train_rs, y_train_r)
cf_train_time = time.time() - start_time

start_time = time.time()
y_pred_cf = ols_cf.predict(X_test_rs)
cf_pred_time = time.time() - start_time

r2_cf = r2_score(y_test_r, y_pred_cf)
rmse_cf = np.sqrt(mean_squared_error(y_test_r, y_pred_cf))
mae_cf = mean_absolute_error(y_test_r, y_pred_cf)

print("=" * 60)
print("CLOSED-FORM OLS RESULTS")
print("=" * 60)
print(f"Training time:   {cf_train_time:.6f} s")
print(f"Prediction time: {cf_pred_time:.6f} s")
print(f"R^2 score:       {r2_cf:.4f}")
print(f"RMSE:            {rmse_cf:.4f}")
print(f"MAE:             {mae_cf:.4f}")

print(f"\nIntercept: {ols_cf.intercept_:.4f}")
print("Coefficients (on scaled features):")
for name, w in zip(housing.feature_names, ols_cf.coef_):
    print(f"  {name:<10} {w:+.4f}")

## 2.2 Gradient-Descent OLS

### Why bother with gradient descent?

The closed-form solution is beautiful but has costs:
- Building and solving $(X^\top X)\,w = X^\top y$ is $O(np^2 + p^3)$ and requires $O(p^2)$ memory.
- For very large $p$ (or streaming data), this is infeasible.

**Gradient descent** iteratively updates $w$ using the gradient of the MSE:

$$\mathcal{L}(w,b) = \frac{1}{2n}\sum_i (y_i - x_i^\top w - b)^2$$
$$\nabla_w \mathcal{L} = -\tfrac{1}{n} X^\top (y - Xw - b)$$
$$\nabla_b \mathcal{L} = -\tfrac{1}{n}\sum_i (y_i - x_i^\top w - b)$$

Each step costs $O(np)$ — linear in both dimensions — at the cost of tuning a learning rate and a convergence criterion.

In [ ]:
class OLSRegressorGD:
    """
    Ordinary Least Squares regression trained with (batch) gradient descent
    on the mean-squared-error objective.
    """

    def __init__(self, learning_rate=0.01, n_iters=1000, tol=1e-6):
        self.learning_rate = learning_rate
        self.n_iters = n_iters
        self.tol = tol

    def fit(self, X_train, y_train):
        X_train = np.asarray(X_train, dtype=float)
        y_train = np.asarray(y_train, dtype=float)
        n, p = X_train.shape

        self.coef_ = np.zeros(p)
        self.intercept_ = 0.0
        self.loss_history_ = []

        for it in range(self.n_iters):
            y_pred = X_train @ self.coef_ + self.intercept_
            resid = y_train - y_pred

            grad_w = -(X_train.T @ resid) / n
            grad_b = -resid.mean()

            self.coef_ -= self.learning_rate * grad_w
            self.intercept_ -= self.learning_rate * grad_b

            if it % 10 == 0:
                loss = 0.5 * np.mean(resid**2)
                self.loss_history_.append(loss)
                if len(self.loss_history_) > 2 and abs(self.loss_history_[-2] - self.loss_history_[-1]) < self.tol:
                    break
        return self

    def predict(self, X):
        return np.asarray(X, dtype=float) @ self.coef_ + self.intercept_

    def score(self, X, y):
        y_pred = self.predict(X)
        ss_res = np.sum((y - y_pred) ** 2)
        ss_tot = np.sum((y - np.mean(y)) ** 2)
        return 1.0 - ss_res / ss_tot


print("OLSRegressorGD class created successfully!")

### Performance comparison: closed-form vs gradient descent

In [ ]:
ols_gd = OLSRegressorGD(learning_rate=0.05, n_iters=2000, tol=1e-8)

start_time = time.time()
ols_gd.fit(X_train_rs, y_train_r)
gd_train_time = time.time() - start_time

start_time = time.time()
y_pred_gd = ols_gd.predict(X_test_rs)
gd_pred_time = time.time() - start_time

r2_gd = r2_score(y_test_r, y_pred_gd)

print("=" * 60)
print("PERFORMANCE COMPARISON")
print("=" * 60)
print(f"\n{'Method':<22} {'Train Time (s)':<15} {'Predict Time (s)':<18} {'R^2':<10}")
print("-" * 60)
print(f"{'Closed-form OLS':<22} {cf_train_time:<15.6f} {cf_pred_time:<18.6f} {r2_cf:<10.4f}")
print(f"{'Gradient-descent OLS':<22} {gd_train_time:<15.6f} {gd_pred_time:<18.6f} {r2_gd:<10.4f}")
print("-" * 60)

plt.figure(figsize=(8, 4))
plt.plot(np.arange(len(ols_gd.loss_history_)) * 10, ols_gd.loss_history_, "o-", markersize=3)
plt.xlabel("Iteration")
plt.ylabel("MSE / 2")
plt.title("Gradient-Descent OLS - Training Loss")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(
    f"\nLoss values recorded by GD: {len(ols_gd.loss_history_)} (one every 10 iterations, budget n_iters={ols_gd.n_iters})."
)
print("Coefficient agreement (max |diff|):", f"{np.max(np.abs(ols_cf.coef_ - ols_gd.coef_)):.6f}")
print("The closed form is exact; GD only gets as close as its tolerance / iteration budget allows.")

### Visualizing the fit: Predicted vs Actual and Residual plot

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

axes[0].scatter(y_test_r, y_pred_cf, alpha=0.4, s=20)
lims = [y_test_r.min(), y_test_r.max()]
axes[0].plot(lims, lims, "r--", lw=2, label="Perfect Prediction")
axes[0].set_xlabel("Actual ($100k)")
axes[0].set_ylabel("Predicted ($100k)")
axes[0].set_title("Closed-form OLS: Predicted vs Actual", fontweight="bold")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

residuals_cf = y_test_r - y_pred_cf
axes[1].scatter(y_pred_cf, residuals_cf, alpha=0.4, s=20)
axes[1].axhline(0, color="r", linestyle="--", lw=2)
axes[1].set_xlabel("Predicted ($100k)")
axes[1].set_ylabel("Residuals")
axes[1].set_title("Residual Plot", fontweight="bold")
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("Good residual plots show:")
print("1. Random scatter around zero (no pattern)")
print("2. Constant variance across all predicted values (homoskedasticity)")
print("3. Most points close to the zero line")

# 3. Linear Regression: Scikit-learn Implementation

Now that we understand how Linear Regression works internally, let's use scikit-learn's optimized implementation. `LinearRegression` uses a numerically stable SVD-based solver under the hood, and exposes a clean, consistent API shared across all regression variants.

In [ ]:
from sklearn.linear_model import LinearRegression

lr_sklearn = LinearRegression(fit_intercept=True)
lr_sklearn.fit(X_train_rs, y_train_r)

y_pred_sklearn = lr_sklearn.predict(X_test_rs)

r2 = r2_score(y_test_r, y_pred_sklearn)
rmse = np.sqrt(mean_squared_error(y_test_r, y_pred_sklearn))
mae = mean_absolute_error(y_test_r, y_pred_sklearn)

print("=" * 60)
print("SCIKIT-LEARN LINEAR REGRESSION RESULTS")
print("=" * 60)
print(f"R^2 score: {r2:.4f}")
print(f"RMSE:      {rmse:.4f}")
print(f"MAE:       {mae:.4f}")

print(f"\nIntercept: {lr_sklearn.intercept_:.4f}")
coef_df = pd.DataFrame({"feature": housing.feature_names, "coef": lr_sklearn.coef_})
coef_df["abs_coef"] = coef_df["coef"].abs()
coef_df = coef_df.sort_values("abs_coef", ascending=False)
print("Coefficients (on scaled features):")
print(coef_df[["feature", "coef"]].to_string(index=False))

print(f"\nMax |sklearn.coef - our closed-form.coef|: {np.max(np.abs(lr_sklearn.coef_ - ols_cf.coef_)):.2e}")

In [ ]:
# Visualize coefficients
fig, ax = plt.subplots(figsize=(10, 5))
colors = ["tab:blue" if c > 0 else "tab:red" for c in coef_df["coef"]]
ax.barh(coef_df["feature"], coef_df["coef"], color=colors, edgecolor="black")
ax.axvline(0, color="black", lw=1)
ax.set_xlabel("Coefficient (standardized features)")
ax.set_title("Linear Regression Coefficients", fontweight="bold")
ax.grid(True, alpha=0.3, axis="x")
plt.tight_layout()
plt.show()

print("\nInterpretation (standardized features):")
print("A coefficient of +0.5 means: 'a 1-std increase in this feature is")
print("associated with a +0.5 ($50k) increase in predicted price,")
print("holding all other features fixed'.")

# 4. Linear Regression: Statsmodels Implementation

Scikit-learn is wonderful for machine learning — predictions, pipelines, cross-validation — but it deliberately does not expose classical **statistical inference** quantities: standard errors, t-statistics, p-values, confidence intervals, adjusted $R^2$, F-tests, etc.

For that, we switch to **statsmodels**, which is designed around the econometric / statistical tradition.

## 4.1 OLS via `statsmodels.api.OLS`

Two important differences from sklearn:

1. You must **manually add a constant (intercept) column** using `sm.add_constant(X)`.
2. The argument order is **`OLS(y, X)`**, not `fit(X, y)`.

In [ ]:
import statsmodels.api as sm

# Add intercept column (statsmodels does NOT do this automatically)
X_train_sm = sm.add_constant(X_train_rs)
X_test_sm = sm.add_constant(X_test_rs)

# Note the argument order: y first, X second
ols_sm = sm.OLS(y_train_r, X_train_sm)
results_sm = ols_sm.fit()

# Full regression summary
print(results_sm.summary(xname=["const"] + list(housing.feature_names)))

In [ ]:
# Individual pieces of the summary are easy to access programmatically
summary_df = pd.DataFrame(
    {
        "coef": results_sm.params,
        "std_err": results_sm.bse,
        "t": results_sm.tvalues,
        "p>|t|": results_sm.pvalues,
        "CI_low": results_sm.conf_int()[:, 0],
        "CI_high": results_sm.conf_int()[:, 1],
    },
    index=["const"] + list(housing.feature_names),
)
print("Coefficient table:")
print(summary_df.round(4))

print(f"\nR^2:            {results_sm.rsquared:.4f}")
print(f"Adjusted R^2:   {results_sm.rsquared_adj:.4f}")
print(f"F-statistic:    {results_sm.fvalue:.2f}")
print(f"Prob (F-stat):  {results_sm.f_pvalue:.2e}")
print(f"AIC:            {results_sm.aic:.2f}")
print(f"BIC:            {results_sm.bic:.2f}")

print("\nSanity check: sklearn vs statsmodels coefficients should match:")
print(f"  Max |diff|           = {np.max(np.abs(results_sm.params[1:] - lr_sklearn.coef_)):.2e}")
print(f"  |diff| for intercept = {abs(results_sm.params[0] - lr_sklearn.intercept_):.2e}")

### Interpreting the statsmodels output

**Coefficient table:**
* `coef`: Point estimate $\hat{w}_j$.
* `std err`: Standard error of $\hat{w}_j$ — how uncertain is the estimate?
* `t`: $\hat{w}_j / \text{SE}(\hat{w}_j)$, the test statistic for $H_0: w_j = 0$.
* `P>|t|`: Two-sided p-value. Small ($<0.05$) → the feature is statistically significant.
* `[0.025, 0.975]`: 95% confidence interval for $w_j$.

**Model-level diagnostics:**
* **R-squared / Adj. R-squared**: variance explained; adjusted version penalizes extra features.
* **F-statistic / Prob(F)**: tests whether *all* coefficients are jointly zero.
* **AIC / BIC**: information criteria for model comparison — lower is better.
* **Durbin-Watson**: tests autocorrelation of residuals (important for time series).
* **Jarque-Bera / Prob(JB)**: tests normality of residuals.
* **Condition number**: a large value suggests multicollinearity (statsmodels prints a warning above 1,000; here the features are standardized, so the number stays moderate).

Use statsmodels when you care about **inference** — “is this coefficient significantly different from zero?” — rather than pure prediction.

In [ ]:
# Diagnostic plots: residuals
y_pred_sm = results_sm.predict(X_test_sm)
residuals_sm = y_test_r - y_pred_sm

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].scatter(y_pred_sm, residuals_sm, alpha=0.4, s=20)
axes[0].axhline(0, color="r", linestyle="--", lw=2)
axes[0].set_xlabel("Fitted values")
axes[0].set_ylabel("Residuals")
axes[0].set_title("Residuals vs Fitted", fontweight="bold")
axes[0].grid(True, alpha=0.3)

from scipy import stats as scipy_stats

scipy_stats.probplot(residuals_sm, dist="norm", plot=axes[1])
axes[1].set_title("Normal Q-Q Plot of Residuals", fontweight="bold")
axes[1].grid(True, alpha=0.3)

axes[2].hist(residuals_sm, bins=40, edgecolor="black", alpha=0.7)
axes[2].axvline(0, color="r", linestyle="--", lw=2)
axes[2].set_xlabel("Residuals")
axes[2].set_ylabel("Frequency")
axes[2].set_title("Residual Distribution", fontweight="bold")
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 4.2 The Formula API (R-style)

Statsmodels also supports an **R-style formula interface** via `statsmodels.formula.api`. This is often the most readable way to fit a model — and it handles categorical variables and interactions gracefully.

In [ ]:
import statsmodels.formula.api as smf

train_df = pd.DataFrame(X_train_rs, columns=housing.feature_names)
train_df["MedHouseVal"] = y_train_r

formula = "MedHouseVal ~ " + " + ".join(housing.feature_names)
print(f"Formula: {formula}\n")

ols_formula = smf.ols(formula=formula, data=train_df).fit()
print(ols_formula.summary().tables[1])

# 5. Regularized Linear Regression

We've seen plain OLS. Now let's explore the three most common regularized variants: **Ridge** (L2), **Lasso** (L1), and **Elastic Net** (L1 + L2).

To make the differences between these methods visible, we'll work on a version of the problem with many correlated / redundant features — that is where regularization really shines.

## 5.1 Building a Challenging Dataset

In [ ]:
from sklearn.preprocessing import PolynomialFeatures

# Expand the feature set with polynomial interactions so we get many correlated features
poly = PolynomialFeatures(degree=2, include_bias=False, interaction_only=False)
X_train_poly = poly.fit_transform(X_train_r)
X_test_poly = poly.transform(X_test_r)

scaler_poly = StandardScaler()
X_train_poly_s = scaler_poly.fit_transform(X_train_poly)
X_test_poly_s = scaler_poly.transform(X_test_poly)

poly_feature_names = poly.get_feature_names_out(housing.feature_names)

print(f"Original feature count:   {X_train_r.shape[1]}")
print(f"Polynomial feature count: {X_train_poly.shape[1]}")

corr = np.corrcoef(X_train_poly_s, rowvar=False)
np.fill_diagonal(corr, 0)
print(f"Mean |corr| between features: {np.mean(np.abs(corr)):.3f}")
print(f"Max  |corr| between features: {np.max(np.abs(corr)):.3f}")

## 5.2 Ridge Regression (L2)

`sklearn.linear_model.Ridge(alpha)` minimizes (docstring: *"Minimizes the objective function: ||y - Xw||^2_2 + alpha * ||w||^2_2"*)

$$\hat{w}_{\text{ridge}} = \arg\min_w \; \|y - Xw\|_2^2 + \alpha\|w\|_2^2$$

Ridge shrinks coefficients **smoothly** toward zero but does not set them exactly to zero. Closed form:
$$\hat{w} = (X^\top X + \alpha I)^{-1} X^\top y$$

Note the **scale of `alpha`**: the residual term is a *sum* over $n$ observations, not a mean. Relative to the mean-loss convention $\tfrac{1}{2n}\|y - Xw\|_2^2 + \tfrac{\lambda}{2}\|w\|_2^2$ (which is what `Lasso` / `ElasticNet` use), `alpha` $= n\lambda$. So `Ridge(alpha=1)` on 3,500 training rows is a very light penalty, and a Ridge fit comparable to `Lasso(alpha=0.01)` needs `alpha = 0.01 * n`.

In [ ]:
from sklearn.linear_model import Ridge

ridge = Ridge(alpha=1.0)
ridge.fit(X_train_poly_s, y_train_r)
y_pred_ridge = ridge.predict(X_test_poly_s)

print("=" * 60)
print("RIDGE RESULTS (alpha=1.0)")
print("=" * 60)
print(f"R^2:  {r2_score(y_test_r, y_pred_ridge):.4f}")
print(f"RMSE: {np.sqrt(mean_squared_error(y_test_r, y_pred_ridge)):.4f}")
print(f"\nNon-zero coefficients: {(ridge.coef_ != 0).sum()} / {len(ridge.coef_)}")
print(f"Max |coef|: {np.max(np.abs(ridge.coef_)):.4f}")

n_train = X_train_poly_s.shape[0]
print(f"\nNote: alpha=1 with n={n_train} corresponds to lambda = alpha/n = {1 / n_train:.2e} in the")
print("mean-loss convention, i.e. almost no shrinkage - compare with the OLS fit below.")

In [ ]:
from sklearn.model_selection import validation_curve
from sklearn.pipeline import make_pipeline

# Sweep alpha on a log scale. The hyperparameter is selected by 5-fold CV on the
# TRAINING set only (the scaler sits inside the pipeline, so it is re-fitted on each
# training fold); the test set is touched exactly once, at the end.
alphas = np.logspace(-4, 4, 40)
pipe_ridge = make_pipeline(StandardScaler(), Ridge())
train_ridge, cv_ridge = validation_curve(
    pipe_ridge, X_train_poly, y_train_r, param_name="ridge__alpha", param_range=alphas, cv=5, scoring="r2", n_jobs=-1
)
train_ridge_mean, cv_ridge_mean, cv_ridge_std = train_ridge.mean(axis=1), cv_ridge.mean(axis=1), cv_ridge.std(axis=1)

# Coefficient paths (fitted on the whole training set)
paths_ridge = np.array([Ridge(alpha=a).fit(X_train_poly_s, y_train_r).coef_ for a in alphas])

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
axes[0].semilogx(alphas, train_ridge_mean, "o-", label="Train R^2 (fold mean)", markersize=4)
axes[0].semilogx(alphas, cv_ridge_mean, "s-", label="CV R^2 (fold mean +/- std)", markersize=4)
axes[0].fill_between(alphas, cv_ridge_mean - cv_ridge_std, cv_ridge_mean + cv_ridge_std, alpha=0.2)
axes[0].set_xlabel("alpha (log scale)")
axes[0].set_ylabel("R^2")
axes[0].set_title("Ridge: R^2 vs alpha (5-fold CV on the training set)", fontweight="bold")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

for j in range(paths_ridge.shape[1]):
    axes[1].semilogx(alphas, paths_ridge[:, j], alpha=0.5, lw=1)
axes[1].axhline(0, color="black", lw=1)
axes[1].set_xlabel("alpha (log scale)")
axes[1].set_ylabel("Coefficient value")
axes[1].set_title("Ridge Coefficient Path", fontweight="bold")
axes[1].grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

best_ix = int(np.argmax(cv_ridge_mean))
best_alpha_ridge = alphas[best_ix]
ridge_best = Ridge(alpha=best_alpha_ridge).fit(X_train_poly_s, y_train_r)
print(
    f"Best alpha by CV: {best_alpha_ridge:.4f}   CV R^2 = {cv_ridge_mean[best_ix]:.4f} (+/- {cv_ridge_std[best_ix]:.4f})"
)
print(f"Test R^2 at the CV-selected alpha (evaluated once): {ridge_best.score(X_test_poly_s, y_test_r):.4f}")

## 5.3 Lasso Regression (L1)

`sklearn.linear_model.Lasso(alpha)` minimizes (this time with the $\tfrac{1}{2n}$ scaling, exactly as in the docstring)

$$\hat{w}_{\text{lasso}} = \arg\min_w \; \tfrac{1}{2n}\|y - Xw\|_2^2 + \alpha\|w\|_1$$

Because of the L1 penalty, Lasso drives many coefficients **exactly to zero** — automatic **feature selection**.

In [ ]:
from sklearn.linear_model import Lasso

lasso = Lasso(alpha=0.01, max_iter=20000)
lasso.fit(X_train_poly_s, y_train_r)
y_pred_lasso = lasso.predict(X_test_poly_s)

n_nonzero = (lasso.coef_ != 0).sum()
print("=" * 60)
print("LASSO RESULTS (alpha=0.01)")
print("=" * 60)
print(f"R^2:  {r2_score(y_test_r, y_pred_lasso):.4f}")
print(f"RMSE: {np.sqrt(mean_squared_error(y_test_r, y_pred_lasso)):.4f}")
print(
    f"\nNon-zero coefficients: {n_nonzero} / {len(lasso.coef_)} (= {n_nonzero / len(lasso.coef_) * 100:.0f}% selected)"
)

lasso_df = pd.DataFrame({"feature": poly_feature_names, "coef": lasso.coef_})
lasso_df = lasso_df[lasso_df["coef"] != 0].copy()
lasso_df["abs_coef"] = lasso_df["coef"].abs()
lasso_df = lasso_df.sort_values("abs_coef", ascending=False).head(10)
print("\nTop selected features:")
print(lasso_df[["feature", "coef"]].to_string(index=False))

In [ ]:
alphas_l = np.logspace(-4, 1, 40)

# Same recipe as for Ridge: alpha chosen by 5-fold CV on the training set, test set used once.
pipe_lasso = make_pipeline(StandardScaler(), Lasso(max_iter=20000))
train_l, cv_l = validation_curve(
    pipe_lasso, X_train_poly, y_train_r, param_name="lasso__alpha", param_range=alphas_l, cv=5, scoring="r2", n_jobs=-1
)
train_l_mean, cv_l_mean, cv_l_std = train_l.mean(axis=1), cv_l.mean(axis=1), cv_l.std(axis=1)

# Coefficient paths and sparsity (fitted on the whole training set)
paths_l = np.array([Lasso(alpha=a, max_iter=20000).fit(X_train_poly_s, y_train_r).coef_ for a in alphas_l])
sparsity = (paths_l != 0).sum(axis=1)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
axes[0].semilogx(alphas_l, train_l_mean, "o-", label="Train R^2 (fold mean)", markersize=4)
axes[0].semilogx(alphas_l, cv_l_mean, "s-", label="CV R^2 (fold mean +/- std)", markersize=4)
axes[0].fill_between(alphas_l, cv_l_mean - cv_l_std, cv_l_mean + cv_l_std, alpha=0.2)
axes[0].set_xlabel("alpha (log scale)")
axes[0].set_ylabel("R^2")
axes[0].set_title("Lasso: R^2 vs alpha (5-fold CV on the training set)", fontweight="bold")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

for j in range(paths_l.shape[1]):
    axes[1].semilogx(alphas_l, paths_l[:, j], alpha=0.5, lw=1)
axes[1].axhline(0, color="black", lw=1)
axes[1].set_xlabel("alpha (log scale)")
axes[1].set_ylabel("Coefficient value")
axes[1].set_title("Lasso Coefficient Path", fontweight="bold")
axes[1].grid(True, alpha=0.3)

axes[2].semilogx(alphas_l, sparsity, "o-", markersize=4, color="tab:green")
axes[2].set_xlabel("alpha (log scale)")
axes[2].set_ylabel("# non-zero coefficients")
axes[2].set_title("Lasso Sparsity", fontweight="bold")
axes[2].grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

best_ix_l = int(np.argmax(cv_l_mean))
best_alpha_lasso = alphas_l[best_ix_l]
lasso_best = Lasso(alpha=best_alpha_lasso, max_iter=20000).fit(X_train_poly_s, y_train_r)
print(f"Best alpha by CV: {best_alpha_lasso:.4f}   CV R^2 = {cv_l_mean[best_ix_l]:.4f} (+/- {cv_l_std[best_ix_l]:.4f})")
print(f"Test R^2 at the CV-selected alpha (evaluated once): {lasso_best.score(X_test_poly_s, y_test_r):.4f}")
print(f"# features selected at the CV-selected alpha: {sparsity[best_ix_l]}")

## 5.4 Elastic Net (L1 + L2)

`sklearn.linear_model.ElasticNet(alpha, l1_ratio)` minimizes (same $\tfrac{1}{2n}$ convention as `Lasso`)

$$\hat{w}_{\text{enet}} = \arg\min_w \; \tfrac{1}{2n}\|y - Xw\|_2^2 + \alpha\rho\|w\|_1 + \tfrac{\alpha(1-\rho)}{2}\|w\|_2^2$$

Elastic Net gets the best of both worlds:
- The L1 part gives **sparsity** (feature selection).
- The L2 part gives **stability** when features are correlated.

In sklearn, `l1_ratio` is our $\rho$.

In [ ]:
from sklearn.linear_model import ElasticNet

enet = ElasticNet(alpha=0.01, l1_ratio=0.5, max_iter=20000)
enet.fit(X_train_poly_s, y_train_r)
y_pred_enet = enet.predict(X_test_poly_s)

print("=" * 60)
print("ELASTIC NET RESULTS (alpha=0.01, l1_ratio=0.5)")
print("=" * 60)
print(f"R^2:  {r2_score(y_test_r, y_pred_enet):.4f}")
print(f"RMSE: {np.sqrt(mean_squared_error(y_test_r, y_pred_enet)):.4f}")
print(f"\nNon-zero coefficients: {(enet.coef_ != 0).sum()} / {len(enet.coef_)}")

## 5.5 Side-by-Side Comparison: OLS vs Ridge vs Lasso vs Elastic Net

To put the three penalties on a **common scale** we use $\lambda = 0.01$ in the mean-loss convention for all of them: `Lasso(alpha=0.01)`, `ElasticNet(alpha=0.01)` and — because `Ridge` scales its penalty by $n$ — `Ridge(alpha=0.01 * n_train)`.

In [ ]:
models_lin = {
    "OLS": LinearRegression(),
    "Ridge (alpha=0.01*n)": Ridge(alpha=0.01 * n_train),
    "Lasso (alpha=0.01)": Lasso(alpha=0.01, max_iter=20000),
    "Elastic Net (alpha=0.01, rho=0.5)": ElasticNet(alpha=0.01, l1_ratio=0.5, max_iter=20000),
}

rows = []
coef_matrix = {}
for name, model in models_lin.items():
    model.fit(X_train_poly_s, y_train_r)
    y_pred = model.predict(X_test_poly_s)
    rows.append(
        {
            "model": name,
            "train_R2": model.score(X_train_poly_s, y_train_r),
            "test_R2": model.score(X_test_poly_s, y_test_r),
            "test_RMSE": np.sqrt(mean_squared_error(y_test_r, y_pred)),
            "n_nonzero": int((model.coef_ != 0).sum()),
            "max_|coef|": np.max(np.abs(model.coef_)),
        }
    )
    coef_matrix[name] = model.coef_

print("=" * 90)
print("MODEL COMPARISON ON POLYNOMIAL-EXPANDED FEATURES")
print("=" * 90)
print(pd.DataFrame(rows).round(4).to_string(index=False))

# Why is the OLS test R^2 so poor? Look at the worst test residual.
ols_resid = y_test_r - models_lin["OLS"].predict(X_test_poly_s)
worst = int(np.argmax(np.abs(ols_resid)))
print(
    f"\nLargest OLS test residual: {ols_resid[worst]:+.1f} ($100k) for a district with AveOccup = {X_test_r[worst, 5]:.1f}"
)
print(
    f"(training-set maximum of AveOccup: {X_train_r[:, 5].max():.1f}). A few districts with extreme AveOccup / AveRooms"
)
print("values lie outside the training range; their squared and interaction terms make OLS extrapolate wildly,")
print("which is exactly the kind of instability regularization is meant to tame.")

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(20, 7), sharey=True)
for ax, (name, vec) in zip(axes, coef_matrix.items()):
    colors = ["tab:blue" if v > 0 else "tab:red" for v in vec]
    ax.barh(range(len(vec)), vec, color=colors, edgecolor="black", alpha=0.8)
    ax.axvline(0, color="black", lw=1)
    ax.set_title(name, fontweight="bold", fontsize=11)
    ax.grid(True, alpha=0.3, axis="x")
axes[0].set_ylabel("Feature index")
plt.suptitle("Coefficients by Model (polynomial features)", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

print("Observation:")
print("- OLS produces large, sometimes wild coefficients because of multicollinearity.")
print("- Ridge shrinks all coefficients smoothly toward zero.")
print("- Lasso zeroes out most coefficients entirely (sparsity).")
print("- Elastic Net is a compromise: some zeros, some mild shrinkage.")

## 5.6 Regularized Linear Regression via Statsmodels

Statsmodels supports Ridge, Lasso, and Elastic Net through the `fit_regularized` method on `OLS`. Its docstring states the objective explicitly:

$$\tfrac{1}{2n}\,\text{RSS} + \alpha\Bigl[(1 - \texttt{L1\_wt})\,\tfrac{1}{2}\|\beta\|_2^2 + \texttt{L1\_wt}\,\|\beta\|_1\Bigr]$$

where $\beta$ is the **full** parameter vector — **including the constant** that you added with `sm.add_constant`. Three things follow:

* `L1_wt` is what sklearn calls `l1_ratio` ($\rho$): `L1_wt=0` → Ridge, `L1_wt=1` → Lasso.
* `alpha` is on the **mean-loss** scale, so it coincides with sklearn's `Lasso` / `ElasticNet` `alpha`, but **not** with `Ridge`'s: `Ridge(alpha=a)` in sklearn (objective $\text{RSS} + a\|w\|_2^2$) equals statsmodels with `alpha = a / n`. Conversely, our `Ridge(alpha=0.01 * n_train)` from Section 5.5 is statsmodels `alpha=0.01`.
* To leave the intercept unpenalized (as sklearn always does) pass `alpha` as a **vector** with a `0` in the position of the constant. If you pass a scalar, the intercept gets shrunk as well — with `L1_wt=0` and centred (e.g. standardized) features it is multiplied by exactly $1/(1+\alpha)$, which can wreck the fit.

Note: for regularized fits, statsmodels does NOT return the classical standard errors / p-values — the sampling distribution of penalized estimators is non-standard. You only get point estimates.

In [ ]:
X_train_poly_sm = sm.add_constant(X_train_poly_s)
X_test_poly_sm = sm.add_constant(X_test_poly_s)

# lambda = 0.01 for all three (mean-loss convention); 0 for the constant so that the
# intercept is not penalized (sklearn never penalizes it).
lam = 0.01
alpha_vec = np.full(X_train_poly_sm.shape[1], lam)
alpha_vec[0] = 0.0

ridge_sm = sm.OLS(y_train_r, X_train_poly_sm).fit_regularized(alpha=alpha_vec, L1_wt=0.0)
lasso_sm = sm.OLS(y_train_r, X_train_poly_sm).fit_regularized(alpha=alpha_vec, L1_wt=1.0, maxiter=2000)
enet_sm = sm.OLS(y_train_r, X_train_poly_sm).fit_regularized(alpha=alpha_vec, L1_wt=0.5, maxiter=2000)


def enet_objective(w, b, lam, rho):
    """The sklearn / statsmodels elastic-net objective (mean-loss convention, intercept unpenalized)."""
    resid = y_train_r - X_train_poly_s @ w - b
    return 0.5 * np.mean(resid**2) + lam * (rho * np.abs(w).sum() + 0.5 * (1 - rho) * (w**2).sum())


print("statsmodels vs sklearn on the polynomial features (same objective, lambda = 0.01)")
print("-" * 78)
pairs = [
    ("Ridge   (sm alpha=0.01  <->  sklearn Ridge(alpha=0.01*n))", ridge_sm, models_lin["Ridge (alpha=0.01*n)"], 0.0),
    ("Lasso   (sm alpha=0.01  <->  sklearn Lasso(alpha=0.01))", lasso_sm, models_lin["Lasso (alpha=0.01)"], 1.0),
    (
        "ENet    (sm alpha=0.01  <->  sklearn ElasticNet(alpha=0.01))",
        enet_sm,
        models_lin["Elastic Net (alpha=0.01, rho=0.5)"],
        0.5,
    ),
]
for label, fit_sm, fit_sk, rho in pairs:
    print(label)
    print(
        f"   max |coef diff| = {np.max(np.abs(fit_sm.params[1:] - fit_sk.coef_)):.2e}   "
        f"|intercept diff| = {abs(fit_sm.params[0] - fit_sk.intercept_):.2e}"
    )
    print(
        f"   objective: statsmodels = {enet_objective(fit_sm.params[1:], fit_sm.params[0], lam, rho):.6f}   "
        f"sklearn = {enet_objective(fit_sk.coef_, fit_sk.intercept_, lam, rho):.6f}"
    )

print("\nPredictive performance on the test set:")
for name, fit in [("Ridge (sm)", ridge_sm), ("Lasso (sm)", lasso_sm), ("Elastic Net (sm)", enet_sm)]:
    y_pred_s = fit.predict(X_test_poly_sm)
    print(
        f"  {name:<18}  R^2 = {r2_score(y_test_r, y_pred_s):.4f}   "
        f"RMSE = {np.sqrt(mean_squared_error(y_test_r, y_pred_s)):.4f}"
    )

**Reading the output.** For Ridge the two libraries solve the same closed-form problem and agree to machine precision — once `alpha` is rescaled by $n$ and the constant is left unpenalized. For Lasso and Elastic Net the *objective* is identical, but the two coordinate-descent solvers stop at different points on this ill-conditioned design (44 highly correlated polynomial features): the objective values printed above show which solver got closer to the optimum. Whenever you compare implementations of a penalized model, compare **objective values**, not just coefficients.

# 6. Logistic Regression: From-Scratch Implementation

We now turn to **logistic regression** — the classification counterpart of linear regression. We'll implement binary logistic regression from scratch using gradient descent on the cross-entropy loss.

Recall:
$$\hat{p}_i = \sigma(x_i^\top w + b) = \frac{1}{1 + e^{-(x_i^\top w + b)}}$$
$$\mathcal{L}(w,b) = -\frac{1}{n}\sum_i\bigl[y_i\log\hat{p}_i + (1-y_i)\log(1-\hat{p}_i)\bigr]$$
$$\nabla_w \mathcal{L} = \tfrac{1}{n} X^\top(\hat{p} - y),\quad \nabla_b\mathcal{L} = \tfrac{1}{n}\sum_i(\hat{p}_i - y_i)$$

In [ ]:
# Load the Breast Cancer dataset (our classification benchmark)
from sklearn.datasets import load_breast_cancer

cancer = load_breast_cancer()
X_clf = cancer.data
y_clf = cancer.target  # 0 = malignant, 1 = benign

print("Breast Cancer Wisconsin Dataset Loaded")
print(f"Features shape: {X_clf.shape}")
print(f"Target shape:   {y_clf.shape}")
print(f"\nClasses: {cancer.target_names}")
print(f"Class distribution: {np.bincount(y_clf)}")
print(f"\nFirst 3 feature names: {cancer.feature_names[:3]}")

### Step 1: Implement sigmoid, loss and gradient

In [ ]:
def sigmoid(z):
    """Numerically stable sigmoid."""
    # For large negative z, exp(-z) overflows; we split the logic to
    # avoid warnings and return a stable result.
    out = np.empty_like(z, dtype=float)
    pos = z >= 0
    out[pos] = 1.0 / (1.0 + np.exp(-z[pos]))
    out[~pos] = np.exp(z[~pos]) / (1.0 + np.exp(z[~pos]))
    return out


def cross_entropy(w, b, X, y):
    """Mean binary cross-entropy loss."""
    p = sigmoid(X @ w + b)
    eps = 1e-12  # avoid log(0)
    return -np.mean(y * np.log(p + eps) + (1 - y) * np.log(1 - p + eps))


def cross_entropy_gradient(w, b, X, y):
    """Gradient of the mean BCE loss w.r.t. w and b."""
    p = sigmoid(X @ w + b)
    err = p - y
    grad_w = (X.T @ err) / len(y)
    grad_b = err.mean()
    return grad_w, grad_b


# Quick sanity check
w_t = np.array([0.5, -0.2])
b_t = 0.1
X_t = np.array([[1.0, 2.0], [-1.0, -1.0], [0.5, 0.8]])
y_t = np.array([1, 0, 1])
print(f"Loss:   {cross_entropy(w_t, b_t, X_t, y_t):.4f}")
gw, gb = cross_entropy_gradient(w_t, b_t, X_t, y_t)
print(f"grad_w: {gw}")
print(f"grad_b: {gb:.4f}")

### Step 2: Implement the Logistic Regression Classifier from Scratch

In [ ]:
class LogisticRegressionScratch:
    """
    Binary logistic regression trained with (batch) gradient descent on the
    cross-entropy loss. Labels must be in {0, 1}.
    """

    def __init__(self, learning_rate=0.1, n_iters=2000, tol=1e-6):
        self.learning_rate = learning_rate
        self.n_iters = n_iters
        self.tol = tol

    def fit(self, X_train, y_train):
        X_train = np.asarray(X_train, dtype=float)
        y_train = np.asarray(y_train, dtype=float)
        n, p = X_train.shape

        self.coef_ = np.zeros(p)
        self.intercept_ = 0.0
        self.loss_history_ = []

        for it in range(self.n_iters):
            gw, gb = cross_entropy_gradient(self.coef_, self.intercept_, X_train, y_train)
            self.coef_ -= self.learning_rate * gw
            self.intercept_ -= self.learning_rate * gb

            if it % 20 == 0:
                loss = cross_entropy(self.coef_, self.intercept_, X_train, y_train)
                self.loss_history_.append(loss)
                if len(self.loss_history_) > 2 and abs(self.loss_history_[-2] - self.loss_history_[-1]) < self.tol:
                    break
        return self

    def decision_function(self, X):
        return np.asarray(X, dtype=float) @ self.coef_ + self.intercept_

    def predict_proba(self, X):
        p1 = sigmoid(self.decision_function(X))
        return np.column_stack([1 - p1, p1])

    def predict(self, X, threshold=0.5):
        return (sigmoid(self.decision_function(X)) >= threshold).astype(int)

    def score(self, X, y):
        return np.mean(self.predict(X) == y)


print("LogisticRegressionScratch class created successfully!")

### Step 3: Split Data and Apply Feature Scaling

Feature scaling is important for logistic regression trained with gradient descent (it accelerates convergence) and is **essential** for the regularized variants we'll see below.

In [ ]:
X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(
    X_clf, y_clf, test_size=0.3, random_state=42, stratify=y_clf
)

print(f"Training set size: {X_train_c.shape[0]}")
print(f"Test set size:     {X_test_c.shape[0]}")
print(f"Train class distribution: {np.bincount(y_train_c)}")
print(f"Test  class distribution: {np.bincount(y_test_c)}")

scaler_c = StandardScaler()
scaler_c.fit(X_train_c)
X_train_cs = scaler_c.transform(X_train_c)
X_test_cs = scaler_c.transform(X_test_c)

### Step 4: Train and evaluate the scratch logistic regression

In [ ]:
from sklearn.metrics import accuracy_score, classification_report, f1_score, precision_score, recall_score

logreg_scratch = LogisticRegressionScratch(learning_rate=0.1, n_iters=3000, tol=1e-8)

start_time = time.time()
logreg_scratch.fit(X_train_cs, y_train_c)
train_time = time.time() - start_time

y_pred_scratch = logreg_scratch.predict(X_test_cs)
acc = accuracy_score(y_test_c, y_pred_scratch)

print(f"Training time: {train_time:.4f} s")
print(f"Accuracy:      {acc:.4f} ({acc * 100:.2f}%)")

plt.figure(figsize=(8, 4))
plt.plot(np.arange(len(logreg_scratch.loss_history_)) * 20, logreg_scratch.loss_history_, "o-", markersize=3)
plt.xlabel("Iteration")
plt.ylabel("Cross-entropy loss")
plt.title("From-Scratch Logistic Regression - Training Loss")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### Step 5: Confusion Matrix and Classification Report

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay, confusion_matrix

cm = confusion_matrix(y_test_c, y_pred_scratch)

fig, ax = plt.subplots(figsize=(8, 6))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=cancer.target_names)
disp.plot(cmap="Blues", ax=ax)
plt.title("Confusion Matrix - Logistic Regression (From Scratch)", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

print(classification_report(y_test_c, y_pred_scratch, target_names=cancer.target_names))

### Step 6: Visualize Decision Boundary (2D)

In [ ]:
# Pick two informative features for a 2D visualization
feat_ix = [0, 1]  # mean radius, mean texture
X_train_2d = X_train_cs[:, feat_ix]
X_test_2d = X_test_cs[:, feat_ix]

logreg_2d = LogisticRegressionScratch(learning_rate=0.1, n_iters=3000)
logreg_2d.fit(X_train_2d, y_train_c)

h = 0.02
x_min, x_max = X_train_2d[:, 0].min() - 1, X_train_2d[:, 0].max() + 1
y_min, y_max = X_train_2d[:, 1].min() - 1, X_train_2d[:, 1].max() + 1
xx, yy = np.meshgrid(np.arange(x_min, x_max, h), np.arange(y_min, y_max, h))

Z_prob = sigmoid(logreg_2d.decision_function(np.c_[xx.ravel(), yy.ravel()])).reshape(xx.shape)

fig, ax = plt.subplots(figsize=(12, 8))
cs = ax.contourf(xx, yy, Z_prob, levels=20, alpha=0.5, cmap="coolwarm")
ax.contour(xx, yy, Z_prob, levels=[0.5], colors="k", linewidths=2)

scatter = ax.scatter(
    X_train_2d[:, 0], X_train_2d[:, 1], c=y_train_c, cmap="coolwarm", edgecolor="black", s=60, alpha=0.8
)
plt.colorbar(cs, ax=ax, label="P(y = benign)")
ax.set_xlabel(f"standardized {cancer.feature_names[feat_ix[0]]}")
ax.set_ylabel(f"standardized {cancer.feature_names[feat_ix[1]]}")
ax.set_title("Logistic Regression - Decision Boundary (2D, From Scratch)", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

acc_2d = logreg_2d.score(X_test_2d, y_test_c)
print(f"Accuracy with only 2 features: {acc_2d:.4f} ({acc_2d * 100:.2f}%)")

# 7. Logistic Regression: Scikit-learn Implementation

Scikit-learn's `LogisticRegression` is the optimized, production-ready implementation. It supports several solvers (`lbfgs`, `liblinear`, `saga`, `newton-cg`), automatic L1/L2/Elastic Net regularization, class weighting, and a consistent API with every other sklearn estimator.

In [ ]:
from sklearn.linear_model import LogisticRegression

# Very large C (=> very weak regularization). The documented spelling for "no penalty"
# is C=np.inf (scikit-learn >= 1.8), but 1.8.0 emits a spurious warning for it, so we
# use a large finite C here.
logreg_skl = LogisticRegression(C=1e6, solver="lbfgs", max_iter=5000)
logreg_skl.fit(X_train_cs, y_train_c)

y_pred_skl = logreg_skl.predict(X_test_cs)
y_prob_skl = logreg_skl.predict_proba(X_test_cs)[:, 1]

acc = accuracy_score(y_test_c, y_pred_skl)
prec = precision_score(y_test_c, y_pred_skl)
rec = recall_score(y_test_c, y_pred_skl)
f1 = f1_score(y_test_c, y_pred_skl)

print("=" * 60)
print("SCIKIT-LEARN LOGISTIC REGRESSION RESULTS")
print("=" * 60)
print(f"Accuracy:  {acc:.4f}")
print(f"Precision: {prec:.4f}")
print(f"Recall:    {rec:.4f}")
print(f"F1-score:  {f1:.4f}")

print(classification_report(y_test_c, y_pred_skl, target_names=cancer.target_names))
print(f"Intercept: {logreg_skl.intercept_[0]:+.4f}")
print(
    f"Agreement with from-scratch coefs (max |diff|): {np.max(np.abs(logreg_skl.coef_[0] - logreg_scratch.coef_)):.4f}"
)
print(f"Max |coef|: {np.max(np.abs(logreg_skl.coef_)):.2f}")
# Note: on all 30 standardized features the training classes are (quasi-)perfectly
# separable, so an unpenalized MLE does not exist: the loss keeps decreasing as the
# coefficients grow, and both solvers simply stop somewhere along the way (hence the
# large coefficients and the imperfect agreement). Section 8 discusses this in detail.

In [ ]:
# ROC curve
from sklearn.metrics import auc, roc_curve

fpr, tpr, _ = roc_curve(y_test_c, y_prob_skl)
roc_auc = auc(fpr, tpr)

fig, ax = plt.subplots(figsize=(7, 6))
ax.plot(fpr, tpr, lw=2, label=f"ROC curve (AUC = {roc_auc:.4f})")
ax.plot([0, 1], [0, 1], "k--", lw=1)
ax.set_xlabel("False Positive Rate")
ax.set_ylabel("True Positive Rate")
ax.set_title("ROC Curve - Sklearn Logistic Regression", fontweight="bold")
ax.legend(loc="lower right")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# 8. Logistic Regression: Statsmodels Implementation

For **inference** on logistic regression we again turn to statsmodels. The model is a **Generalized Linear Model (GLM)** with a Binomial family and a logit link. We have two equivalent APIs:

1. **`sm.Logit(y, X)`** — the direct logistic regression model.
2. **`sm.GLM(y, X, family=sm.families.Binomial())`** — the general GLM interface.

Both give the same coefficients, p-values, and confidence intervals. Remember: `sm.add_constant(X)` is required (statsmodels does not add an intercept for you).

**Which features?** Classical inference needs a maximum-likelihood estimate that actually *exists*. On all 30 standardized breast-cancer features the training classes can be separated perfectly by a hyperplane, so the MLE runs off to infinity (we show this failure below). For the inference demo we therefore use a small, deliberately "weaker" subset of five shape features on which the fit is well-behaved.

In [ ]:
# A small, non-separable design for the inference demo (see the note below)
inference_features = ["mean texture", "mean smoothness", "mean compactness", "mean symmetry", "mean fractal dimension"]
inference_ix = [list(cancer.feature_names).index(f) for f in inference_features]

X_train_c_sm = sm.add_constant(X_train_cs[:, inference_ix])
X_test_c_sm = sm.add_constant(X_test_cs[:, inference_ix])

# Option 1: sm.Logit (default solver: Newton-Raphson)
logit_sm = sm.Logit(y_train_c, X_train_c_sm)
res_logit = logit_sm.fit(disp=False)

print(res_logit.summary(xname=["const"] + inference_features))

In [ ]:
# Option 2: sm.GLM with a Binomial family (equivalent; fitted by IRLS)
glm_sm = sm.GLM(y_train_c, X_train_c_sm, family=sm.families.Binomial())
res_glm = glm_sm.fit()
print("sm.Logit vs sm.GLM agreement:")
print(f"  Max |coef diff|    = {np.max(np.abs(res_logit.params - res_glm.params)):.2e}")
print(f"  Max |std err diff| = {np.max(np.abs(res_logit.bse - res_glm.bse)):.2e}")

# Build a clean coefficient table with odds ratios
coef_table = pd.DataFrame(
    {
        "coef (log-odds)": res_logit.params,
        "std_err": res_logit.bse,
        "z": res_logit.tvalues,
        "p>|z|": res_logit.pvalues,
        "CI_low": res_logit.conf_int()[:, 0],
        "CI_high": res_logit.conf_int()[:, 1],
        "odds_ratio": np.exp(res_logit.params),
    },
    index=["const"] + inference_features,
)

print("\nCoefficient table (sorted by |z|):")
print(
    coef_table.assign(abs_z=coef_table["z"].abs()).sort_values("abs_z", ascending=False).drop(columns="abs_z").round(4)
)

# Pseudo-R^2 (McFadden's) and model statistics
print(f"\nConverged:                {res_logit.mle_retvals['converged']}")
print(f"Log-likelihood:           {res_logit.llf:.2f}")
print(f"LL-Null (intercept-only): {res_logit.llnull:.2f}")
print(f"Pseudo R^2 (McFadden):    {res_logit.prsquared:.4f}")
print(f"LLR p-value:              {res_logit.llr_pvalue:.2e}")
print(f"AIC:                      {res_logit.aic:.2f}")
print(f"BIC:                      {res_logit.bic:.2f}")

### Interpreting logistic regression coefficients

In logistic regression each coefficient $w_j$ is on the **log-odds** scale:

* **coef = 0** → feature has no effect on the odds.
* **coef > 0** → feature increases the probability of class 1.
* **odds_ratio = $e^{w_j}$** → a one-unit (here: 1-std, since we standardized) increase in the feature multiplies the odds of class 1 by $e^{w_j}$.

For example, `odds_ratio = 2.0` means: “a one-unit increase in this feature doubles the odds of class 1, holding everything else fixed”.

**Model-level diagnostics:**
* **Pseudo R² (McFadden)**: $1 - \tfrac{\text{LL}_\text{full}}{\text{LL}_\text{null}}$. Not comparable to linear-regression R²: in typical social-science / choice models values of 0.2–0.4 already indicate an excellent fit (McFadden's own rule of thumb). The value above is much higher than that because the two tumour classes are unusually well separated — and it would approach 1 on a separable design, which is a warning sign rather than a success (see below).
* **LLR p-value**: Likelihood-ratio test for "all coefficients are zero".
* **AIC / BIC**: Information criteria for model comparison.

### Prediction with statsmodels

`res_logit.predict(X)` returns **probabilities** (not hard labels). Threshold at 0.5 (or a different value based on your cost function) to get class predictions.

In [ ]:
# Prediction using statsmodels
y_prob_sm = res_logit.predict(X_test_c_sm)
y_pred_sm_bin = (y_prob_sm >= 0.5).astype(int)

print(f"statsmodels accuracy (5 features): {accuracy_score(y_test_c, y_pred_sm_bin):.4f}")

# Cross-check: an (essentially) unpenalized sklearn fit on the same 5 features must
# reproduce the MLE. (C=np.inf is the documented spelling from sklearn 1.9 on.)
logreg_skl_5 = LogisticRegression(C=1e10, solver="lbfgs", tol=1e-8, max_iter=5000)
logreg_skl_5.fit(X_train_cs[:, inference_ix], y_train_c)
print(f"Max |coef diff| statsmodels vs sklearn: {np.max(np.abs(logreg_skl_5.coef_[0] - res_logit.params[1:])):.2e}")
print(
    f"Agreement of test predictions:          {np.mean(y_pred_sm_bin == logreg_skl_5.predict(X_test_cs[:, inference_ix])):.4f}"
)

### ⚠️ A note on perfect (quasi-)separation

**What it is.** If some hyperplane $x^\top w + b = 0$ puts every $y=1$ on one side and every $y=0$ on the other (*complete separation*), or does so for all but a few observations lying exactly on the boundary (*quasi-complete separation*), then scaling $w$ up makes every fitted probability move towards 0 or 1 and the log-likelihood towards 0. The likelihood has **no finite maximizer**: the MLE does not exist, and neither do standard errors, p-values or confidence intervals.

**How to recognize it in practice:**
* coefficients that explode (hundreds or thousands on standardized features) and keep growing with more iterations,
* `PerfectSeparationWarning` / "Possibly complete quasi-separation" from statsmodels, `overflow encountered in exp`, `HessianInversionWarning`, or a `LinAlgError: Singular matrix` from the Newton solver,
* `NaN` standard errors and p-values, odds ratios of `inf` / `0`,
* McFadden pseudo-$R^2 \approx 1$ (log-likelihood $\approx 0$) — too good to be true.

**Why regularization (or fewer features) fixes it.** An L2 (or L1) penalty adds $\lambda\|w\|^2$ to the objective, so the penalized loss *does* have a finite minimizer; that is why `sklearn`'s `LogisticRegression` — regularized by default — never crashes on this data. Alternatives are Firth's penalized likelihood, a Bayesian prior on $w$, or dropping features until the classes overlap (what we did above). Remember, though, that penalized coefficients no longer come with the classical p-values.

**Why it matters in finance.** With many features and few events — corporate defaults, fraud cases, rating downgrades — it is easy for some combination of features to separate the handful of "1"s perfectly in the training sample. The result looks like a perfect model but is pure overfitting. Regularization, cross-validation and a healthy suspicion of pseudo-$R^2 \approx 1$ are the antidotes.

Let us reproduce the failure on the full 30-feature design (warnings are captured so that we can inspect them).

In [ ]:
# Deliberately fit the unpenalized model on all 30 standardized features with three solvers.
X_train_all_sm = sm.add_constant(X_train_cs)

with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    try:
        res_sep = sm.Logit(y_train_c, X_train_all_sm).fit(method="bfgs", maxiter=200, disp=False)
        print("Fit on 30 features (BFGS):")
        print(
            f"  max |coef|             = {np.max(np.abs(res_sep.params)):.1f}   (5-feature model: {np.max(np.abs(res_logit.params)):.2f})"
        )
        print(f"  NaN standard errors    = {np.isnan(res_sep.bse).sum()} / {len(res_sep.bse)}")
        print(f"  log-likelihood         = {res_sep.llf:.4f}   pseudo R^2 = {res_sep.prsquared:.4f}")
        print(f"  odds ratios: min = {np.exp(res_sep.params).min():.2e}, max = {np.exp(res_sep.params).max():.2e}")
    except Exception as err:  # we want to show whatever goes wrong
        print(f"Fit on 30 features failed: {type(err).__name__}: {err}")

    try:
        sm.Logit(y_train_c, X_train_all_sm).fit(disp=False)  # default Newton-Raphson
        print("Newton-Raphson: finished")
    except Exception as err:
        print(f"Newton-Raphson: {type(err).__name__}: {err}")

    try:
        res_sep_glm = sm.GLM(y_train_c, X_train_all_sm, family=sm.families.Binomial()).fit()  # IRLS
        print(f"GLM / IRLS: finished, max |coef| = {np.max(np.abs(res_sep_glm.params)):.1f}")
    except Exception as err:
        print(f"GLM / IRLS: {type(err).__name__}: {err}")

print(f"\nWarnings raised during the three fits ({len(caught)} in total):")
for name in sorted({w.category.__name__ for w in caught}):
    print(f"  - {name}")

# The cure: any amount of regularization gives a finite, well-defined solution.
l2_fit = LogisticRegression(C=1.0, solver="lbfgs", max_iter=5000).fit(X_train_cs, y_train_c)
print(f"\nsklearn LogisticRegression(C=1.0) on the same 30 features: max |coef| = {np.max(np.abs(l2_fit.coef_)):.2f}")

## 8.1 The Formula API (R-style)

Just like `smf.ols`, statsmodels exposes `smf.logit` / `smf.glm` with the formula interface. This is particularly handy when you have categorical variables — statsmodels will dummy-encode them automatically.

In [ ]:
# Same five features as in the inference demo above
feature_subset = inference_features

train_df_c = pd.DataFrame(X_train_cs, columns=cancer.feature_names)
train_df_c["target"] = y_train_c

# Remember to sanitize feature names with spaces for the formula interface.
safe_names = [f.replace(" ", "_") for f in feature_subset]
train_df_c_renamed = train_df_c.rename(columns=dict(zip(feature_subset, safe_names)))

formula = "target ~ " + " + ".join(safe_names)
print(f"Formula: {formula}\n")

logit_formula = smf.logit(formula=formula, data=train_df_c_renamed).fit(disp=False)
print(logit_formula.summary().tables[1])

# 9. Regularized Logistic Regression

Just like linear regression, logistic regression can be regularized with L1, L2, or Elastic Net penalties. In the mean-loss convention of Section 1 the objective becomes:

$$\mathcal{L}(w,b) = \frac{1}{n}\sum_{i=1}^n \ell_i(w,b) + \lambda\,\Omega(w)$$

where $\ell_i$ is the cross-entropy of observation $i$ and $\Omega(w)$ is the penalty (L1, L2, or a mix).

**Sklearn convention** (`LogisticRegression` docstring): the solver minimizes

$$C \sum_{i=1}^n \ell_i(w,b) + \rho\|w\|_1 + \tfrac{1-\rho}{2}\|w\|_2^2, \qquad \rho = \texttt{l1\_ratio}.$$

Dividing by $Cn$ shows that this is the mean-loss objective with $\lambda = 1/(Cn)$: large `C` = weak regularization, small `C` = heavy regularization, and `C=np.inf` = no penalty. The mixing parameter `l1_ratio` selects the penalty (since scikit-learn 1.8 the old `penalty=` argument is deprecated and will be removed in 1.10):

| `l1_ratio` | Penalty | Compatible `solver` |
|---|---|---|
| `0.0` (default) | L2 | `lbfgs` (default), `newton-cg`, `newton-cholesky`, `sag`, `saga`, `liblinear` |
| `1.0` | L1 | `liblinear`, `saga` |
| `0 < l1_ratio < 1` | Elastic Net | `saga` only |
| — (`C=np.inf`) | none | `lbfgs`, `newton-cg`, `newton-cholesky`, `sag`, `saga` |

## 9.1 L2-Regularized Logistic Regression (default)

In [ ]:
l2_logreg = LogisticRegression(C=1.0, solver="lbfgs", max_iter=5000)
l2_logreg.fit(X_train_cs, y_train_c)
y_pred_l2 = l2_logreg.predict(X_test_cs)

print("=" * 60)
print("L2-PENALIZED LOGISTIC REGRESSION (C=1.0)")
print("=" * 60)
print(f"Accuracy: {accuracy_score(y_test_c, y_pred_l2):.4f}")
print(f"F1:       {f1_score(y_test_c, y_pred_l2):.4f}")
print(f"Non-zero coefficients: {(l2_logreg.coef_[0] != 0).sum()} / {X_train_cs.shape[1]}")

## 9.2 L1-Regularized Logistic Regression (feature selection)

In [ ]:
# L1 needs 'liblinear' or 'saga'; we use saga (it also handles Elastic Net below)
l1_logreg = LogisticRegression(C=0.1, l1_ratio=1.0, solver="saga", max_iter=5000)
l1_logreg.fit(X_train_cs, y_train_c)
y_pred_l1 = l1_logreg.predict(X_test_cs)

print("=" * 60)
print("L1-PENALIZED LOGISTIC REGRESSION (C=0.1)")
print("=" * 60)
print(f"Accuracy: {accuracy_score(y_test_c, y_pred_l1):.4f}")
print(f"F1:       {f1_score(y_test_c, y_pred_l1):.4f}")
n_nonzero_l1 = (l1_logreg.coef_[0] != 0).sum()
print(
    f"Non-zero coefficients: {n_nonzero_l1} / {X_train_cs.shape[1]} "
    f"(= {n_nonzero_l1 / X_train_cs.shape[1] * 100:.0f}% selected)"
)

l1_df = pd.DataFrame({"feature": cancer.feature_names, "coef": l1_logreg.coef_[0]})
l1_df = l1_df[l1_df["coef"] != 0].copy()
l1_df["abs_coef"] = l1_df["coef"].abs()
l1_df = l1_df.sort_values("abs_coef", ascending=False).head(10)
print("\nTop selected features:")
print(l1_df[["feature", "coef"]].to_string(index=False))

## 9.3 Elastic Net Logistic Regression

In [ ]:
# saga solver supports elasticnet
enet_logreg = LogisticRegression(C=0.5, l1_ratio=0.5, solver="saga", max_iter=10000)
enet_logreg.fit(X_train_cs, y_train_c)
y_pred_enet_l = enet_logreg.predict(X_test_cs)

print("=" * 60)
print("ELASTIC NET LOGISTIC REGRESSION (C=0.5, l1_ratio=0.5)")
print("=" * 60)
print(f"Accuracy: {accuracy_score(y_test_c, y_pred_enet_l):.4f}")
print(f"F1:       {f1_score(y_test_c, y_pred_enet_l):.4f}")
print(f"Non-zero coefficients: {(enet_logreg.coef_[0] != 0).sum()} / {X_train_cs.shape[1]}")

## 9.4 Coefficient Paths: C Sweep

In [ ]:
Cs = np.logspace(-3, 3, 30)

# Coefficient paths (fitted on the whole training set)
paths_l2 = np.array(
    [LogisticRegression(C=C, solver="lbfgs", max_iter=5000).fit(X_train_cs, y_train_c).coef_[0] for C in Cs]
)
paths_l1 = np.array(
    [
        LogisticRegression(C=C, l1_ratio=1.0, solver="saga", max_iter=5000).fit(X_train_cs, y_train_c).coef_[0]
        for C in Cs
    ]
)

# C selected by 5-fold CV on the TRAINING set (scaler inside the pipeline); test set used once at the end.
pipe_l2 = make_pipeline(StandardScaler(), LogisticRegression(solver="lbfgs", max_iter=5000))
pipe_l1 = make_pipeline(StandardScaler(), LogisticRegression(l1_ratio=1.0, solver="saga", max_iter=5000))
_, cv_acc_l2 = validation_curve(
    pipe_l2,
    X_train_c,
    y_train_c,
    param_name="logisticregression__C",
    param_range=Cs,
    cv=5,
    scoring="accuracy",
    n_jobs=-1,
)
_, cv_acc_l1 = validation_curve(
    pipe_l1,
    X_train_c,
    y_train_c,
    param_name="logisticregression__C",
    param_range=Cs,
    cv=5,
    scoring="accuracy",
    n_jobs=-1,
)
cv_l2_mean, cv_l2_std = cv_acc_l2.mean(axis=1), cv_acc_l2.std(axis=1)
cv_l1_mean, cv_l1_std = cv_acc_l1.mean(axis=1), cv_acc_l1.std(axis=1)

fig, axes = plt.subplots(2, 2, figsize=(16, 10))

for j in range(paths_l2.shape[1]):
    axes[0, 0].semilogx(Cs, paths_l2[:, j], alpha=0.5, lw=1)
axes[0, 0].axhline(0, color="black", lw=1)
axes[0, 0].set_title("L2 coefficient path", fontweight="bold")
axes[0, 0].set_xlabel("C (log scale)")
axes[0, 0].set_ylabel("Coefficient")
axes[0, 0].grid(True, alpha=0.3)

for j in range(paths_l1.shape[1]):
    axes[0, 1].semilogx(Cs, paths_l1[:, j], alpha=0.5, lw=1)
axes[0, 1].axhline(0, color="black", lw=1)
axes[0, 1].set_title("L1 coefficient path", fontweight="bold")
axes[0, 1].set_xlabel("C (log scale)")
axes[0, 1].set_ylabel("Coefficient")
axes[0, 1].grid(True, alpha=0.3)

axes[1, 0].semilogx(Cs, cv_l2_mean, "o-", label="L2", markersize=4)
axes[1, 0].fill_between(Cs, cv_l2_mean - cv_l2_std, cv_l2_mean + cv_l2_std, alpha=0.2)
axes[1, 0].semilogx(Cs, cv_l1_mean, "s-", label="L1", markersize=4)
axes[1, 0].fill_between(Cs, cv_l1_mean - cv_l1_std, cv_l1_mean + cv_l1_std, alpha=0.2)
axes[1, 0].set_title("5-fold CV accuracy (training set) vs C", fontweight="bold")
axes[1, 0].set_xlabel("C (log scale)")
axes[1, 0].set_ylabel("Accuracy")
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

sparsity_l1 = (paths_l1 != 0).sum(axis=1)
axes[1, 1].semilogx(Cs, sparsity_l1, "o-", markersize=4, color="tab:green")
axes[1, 1].set_title("L1 sparsity (# non-zero coefficients) vs C", fontweight="bold")
axes[1, 1].set_xlabel("C (log scale)")
axes[1, 1].set_ylabel("# non-zero")
axes[1, 1].grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# np.argmax returns the FIRST (i.e. smallest C = most regularized) maximizer in case of ties.
best_ix_l2, best_ix_l1 = int(np.argmax(cv_l2_mean)), int(np.argmax(cv_l1_mean))
best_l2 = LogisticRegression(C=Cs[best_ix_l2], solver="lbfgs", max_iter=5000).fit(X_train_cs, y_train_c)
best_l1 = LogisticRegression(C=Cs[best_ix_l1], l1_ratio=1.0, solver="saga", max_iter=5000).fit(X_train_cs, y_train_c)
print(
    f"L2: best C by CV = {Cs[best_ix_l2]:.4f}   CV acc = {cv_l2_mean[best_ix_l2]:.4f}   "
    f"test acc (once) = {best_l2.score(X_test_cs, y_test_c):.4f}"
)
print(
    f"L1: best C by CV = {Cs[best_ix_l1]:.4f}   CV acc = {cv_l1_mean[best_ix_l1]:.4f}   "
    f"test acc (once) = {best_l1.score(X_test_cs, y_test_c):.4f}   "
    f"non-zero coefs = {sparsity_l1[best_ix_l1]}"
)

## 9.5 Regularized Logistic Regression via Statsmodels

Two different statsmodels entry points — with two different `alpha` scalings:

* **`sm.Logit(y, X).fit_regularized(alpha=...)`** supports **L1 only** (the `method` argument accepts `'l1'` / `'l1_cvxopt_cp'` — an `L1_wt` keyword is silently ignored, so "Ridge" and "Elastic Net" fits made this way are just Lasso fits). Its objective is written in terms of the **summed** negative log-likelihood, $-\ell(\beta) + \sum_k \alpha_k|\beta_k|$, so it matches sklearn's `LogisticRegression(C, l1_ratio=1)` (objective $C\sum_i \ell_i + \|w\|_1$) with $\alpha = 1/C$.
* **`sm.GLM(y, X, family=sm.families.Binomial()).fit_regularized(alpha=..., L1_wt=...)`** supports the full **Elastic Net** (`L1_wt` = sklearn's `l1_ratio`). Its objective uses the **mean** log-likelihood, $-\ell(\beta)/n + \alpha\bigl[(1-\texttt{L1\_wt})\tfrac{1}{2}\|\beta\|_2^2 + \texttt{L1\_wt}\|\beta\|_1\bigr]$, so the same sklearn model corresponds to $\alpha = 1/(Cn)$.

In both cases pass `alpha` as a **vector with 0 for the constant**, otherwise the intercept is penalized too (sklearn never penalizes it). Below we verify the two mappings numerically against sklearn.

Again: regularized fits in statsmodels do **not** return p-values / standard errors.

In [ ]:
C_sm = 0.1
X_train_all_sm = sm.add_constant(X_train_cs)  # all 30 features (+ constant)
X_test_all_sm = sm.add_constant(X_test_cs)
n_c, p_c = X_train_all_sm.shape

# sm.Logit: L1 only, alpha = 1/C (sum log-likelihood), 0 for the constant
alpha_logit = np.full(p_c, 1.0 / C_sm)
alpha_logit[0] = 0.0
logit_l1_sm = sm.Logit(y_train_c, X_train_all_sm).fit_regularized(alpha=alpha_logit, disp=False, maxiter=2000)

# sm.GLM: Elastic Net, alpha = 1/(C n) (mean log-likelihood), 0 for the constant
alpha_glm = np.full(p_c, 1.0 / (C_sm * n_c))
alpha_glm[0] = 0.0
binom = sm.families.Binomial()
glm_l1_sm = sm.GLM(y_train_c, X_train_all_sm, family=binom).fit_regularized(alpha=alpha_glm, L1_wt=1.0, maxiter=5000)
glm_l2_sm = sm.GLM(y_train_c, X_train_all_sm, family=binom).fit_regularized(alpha=alpha_glm, L1_wt=0.0, maxiter=5000)
glm_enet_sm = sm.GLM(y_train_c, X_train_all_sm, family=binom).fit_regularized(alpha=alpha_glm, L1_wt=0.5, maxiter=5000)

# sklearn counterparts with the same C (tight tolerances so that the comparison is meaningful)
skl_l1 = LogisticRegression(C=C_sm, l1_ratio=1.0, solver="saga", tol=1e-8, max_iter=20000).fit(X_train_cs, y_train_c)
skl_l2 = LogisticRegression(C=C_sm, solver="lbfgs", tol=1e-10, max_iter=20000).fit(X_train_cs, y_train_c)
skl_enet = LogisticRegression(C=C_sm, l1_ratio=0.5, solver="saga", tol=1e-8, max_iter=20000).fit(X_train_cs, y_train_c)


def eval_sm(fit, X_test_full):
    prob = fit.predict(X_test_full)
    pred = (prob >= 0.5).astype(int)
    return accuracy_score(y_test_c, pred), f1_score(y_test_c, pred), (fit.params[1:] != 0).sum()


print(
    f"statsmodels vs sklearn, C = {C_sm} (alpha_Logit = 1/C = {1 / C_sm:.3f}, alpha_GLM = 1/(C n) = {1 / (C_sm * n_c):.5f})"
)
print("-" * 95)
comparisons = [
    ("Logit L1   (sm)", logit_l1_sm, skl_l1),
    ("GLM   L1   (sm)", glm_l1_sm, skl_l1),
    ("GLM   L2   (sm)", glm_l2_sm, skl_l2),
    ("GLM   ENet (sm)", glm_enet_sm, skl_enet),
]
for name, fit, skl in comparisons:
    acc_sm, f1_sm, nz = eval_sm(fit, X_test_all_sm)
    print(
        f"{name:<18}  acc={acc_sm:.4f}  F1={f1_sm:.4f}  non-zero coefs={nz:>2}/{p_c - 1} "
        f"(sklearn: {(skl.coef_[0] != 0).sum():>2})   max |coef diff| vs sklearn = {np.max(np.abs(fit.params[1:] - skl.coef_[0])):.1e}"
    )

print("\nThe three GLM fits really are different models:")
print(
    f"  max |L1 - L2| = {np.max(np.abs(glm_l1_sm.params - glm_l2_sm.params)):.3f}   "
    f"max |L1 - ENet| = {np.max(np.abs(glm_l1_sm.params - glm_enet_sm.params)):.3f}"
)

## 9.6 Side-by-Side Comparison: Unpenalized vs L1 vs L2 vs Elastic Net

In [ ]:
models_logit = {
    "Unpenalized (C=1e6, lbfgs)": LogisticRegression(C=1e6, solver="lbfgs", max_iter=5000),
    "L2 (C=1, lbfgs)": LogisticRegression(C=1.0, solver="lbfgs", max_iter=5000),
    "L1 (C=0.1, saga)": LogisticRegression(C=0.1, l1_ratio=1.0, solver="saga", max_iter=5000),
    "Elastic Net (C=0.5, rho=0.5, saga)": LogisticRegression(C=0.5, l1_ratio=0.5, solver="saga", max_iter=10000),
}

rows_l = []
coef_matrix_l = {}
for name, m in models_logit.items():
    m.fit(X_train_cs, y_train_c)
    y_pred = m.predict(X_test_cs)
    y_prob = m.predict_proba(X_test_cs)[:, 1]
    rows_l.append(
        {
            "model": name,
            "accuracy": accuracy_score(y_test_c, y_pred),
            "precision": precision_score(y_test_c, y_pred),
            "recall": recall_score(y_test_c, y_pred),
            "f1": f1_score(y_test_c, y_pred),
            "auc": auc(*roc_curve(y_test_c, y_prob)[:2]),
            "n_nonzero": int((m.coef_[0] != 0).sum()),
        }
    )
    coef_matrix_l[name] = m.coef_[0]

print("=" * 95)
print("LOGISTIC REGRESSION VARIANT COMPARISON")
print("=" * 95)
print(pd.DataFrame(rows_l).round(4).to_string(index=False))

In [ ]:
# Visualize coefficients side-by-side
fig, axes = plt.subplots(1, 4, figsize=(20, 8), sharey=True)
for ax, (name, vec) in zip(axes, coef_matrix_l.items()):
    colors = ["tab:blue" if v > 0 else "tab:red" for v in vec]
    ax.barh(range(len(vec)), vec, color=colors, edgecolor="black", alpha=0.8)
    ax.axvline(0, color="black", lw=1)
    ax.set_title(name, fontweight="bold", fontsize=10)
    ax.grid(True, alpha=0.3, axis="x")
axes[0].set_ylabel("Feature index")
plt.suptitle("Logistic Regression Coefficients by Penalty", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

# 10. Cross-Validation for Model Evaluation

Cross-validation is a crucial technique for assessing how well a model generalizes to unseen data. It helps us avoid overfitting and provides more reliable performance estimates than a single train-test split.

## Why Cross-Validation?

**Problem with single train-test split:**
- Results can vary significantly depending on which samples end up in training vs. test
- May get lucky (or unlucky) with the split
- Doesn't fully utilize the available data for training

**Solution: Cross-Validation**
- Systematically rotate through different train-test combinations
- More robust performance estimate
- Better use of limited data
- Helps detect overfitting / underfitting

## 10.1 Hold-out Approach (Baseline)

In [ ]:
print("=" * 60)
print("HOLD-OUT APPROACH")
print("=" * 60)

# Regression hold-out
lr_ho = LinearRegression().fit(X_train_rs, y_train_r)
print(f"[Regression]     Test R^2: {lr_ho.score(X_test_rs, y_test_r):.4f}")

# Classification hold-out
lg_ho = LogisticRegression(C=1.0, solver="lbfgs", max_iter=5000).fit(X_train_cs, y_train_c)
print(f"[Classification] Test accuracy: {lg_ho.score(X_test_cs, y_test_c):.4f}")

print("\nLIMITATIONS OF HOLD-OUT")
print("1. Performance estimate depends on the specific split")
print("2. Some data is never used for training")
print("3. Can give overly optimistic or pessimistic estimates")
print("4. Not ideal for small datasets")

## 10.2 K-Fold Cross-Validation (Manual)

**How K-Fold Cross-Validation Works:**
1. Split the dataset into K equal-sized folds.
2. For each of the K iterations:
   - Use 1 fold as test, the remaining K-1 folds as train.
   - Train and evaluate the model.
3. Average the K performance scores.

For classification, use **StratifiedKFold** to keep class balance stable across folds.

In [ ]:
from sklearn.model_selection import KFold, StratifiedKFold

print("=" * 60)
print("K-FOLD CV (MANUAL) - REGRESSION (Ridge)")
print("=" * 60)

kf = KFold(n_splits=5, shuffle=True, random_state=42)
fold_scores_reg = []
for i, (tr, te) in enumerate(kf.split(X_reg), 1):
    X_tr, X_te = X_reg[tr], X_reg[te]
    y_tr, y_te = y_reg[tr], y_reg[te]
    sc = StandardScaler()
    m = Ridge(alpha=1.0).fit(sc.fit_transform(X_tr), y_tr)
    s = m.score(sc.transform(X_te), y_te)
    fold_scores_reg.append(s)
    print(f"Fold {i}: R^2 = {s:.4f}")
print(f"\nMean R^2: {np.mean(fold_scores_reg):.4f} (+/- {np.std(fold_scores_reg):.4f})")

print("\n" + "=" * 60)
print("STRATIFIED K-FOLD CV (MANUAL) - CLASSIFICATION (L2 logistic)")
print("=" * 60)

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
fold_scores_clf = []
for i, (tr, te) in enumerate(skf.split(X_clf, y_clf), 1):
    X_tr, X_te = X_clf[tr], X_clf[te]
    y_tr, y_te = y_clf[tr], y_clf[te]
    sc = StandardScaler()
    m = LogisticRegression(C=1.0, solver="lbfgs", max_iter=5000).fit(sc.fit_transform(X_tr), y_tr)
    s = m.score(sc.transform(X_te), y_te)
    fold_scores_clf.append(s)
    print(f"Fold {i}: accuracy = {s:.4f}")
print(f"\nMean accuracy: {np.mean(fold_scores_clf):.4f} (+/- {np.std(fold_scores_clf):.4f})")

## 10.3 Using `cross_validate()`

Sklearn provides `cross_validate()` which automates cross-validation and can compute multiple metrics simultaneously. It also returns both training and test scores.

Pipelines ensure preprocessing (scaling) is fit only on the training portion of each fold — preventing data leakage.

In [ ]:
from sklearn.model_selection import cross_validate
from sklearn.pipeline import Pipeline

# Regression pipeline
pipe_reg = Pipeline([("scaler", StandardScaler()), ("ridge", Ridge(alpha=1.0))])
cv_reg = cross_validate(
    pipe_reg,
    X_reg,
    y_reg,
    cv=5,
    scoring={"r2": "r2", "rmse": "neg_root_mean_squared_error"},
    return_train_score=True,
    n_jobs=-1,
)
print("REGRESSION CROSS_VALIDATE() RESULTS")
print("-" * 40)
print(f"Test R^2:  {np.mean(cv_reg['test_r2']):.4f} (+/- {np.std(cv_reg['test_r2']):.4f})")
print(f"Test RMSE: {-np.mean(cv_reg['test_rmse']):.4f} (+/- {np.std(cv_reg['test_rmse']):.4f})")

# Classification pipeline
pipe_clf = Pipeline(
    [("scaler", StandardScaler()), ("logreg", LogisticRegression(C=1.0, solver="lbfgs", max_iter=5000))]
)
cv_clf = cross_validate(
    pipe_clf,
    X_clf,
    y_clf,
    cv=5,
    scoring={"accuracy": "accuracy", "f1": "f1", "auc": "roc_auc"},
    return_train_score=True,
    n_jobs=-1,
)
print("\nCLASSIFICATION CROSS_VALIDATE() RESULTS")
print("-" * 40)
print(f"Test accuracy: {np.mean(cv_clf['test_accuracy']):.4f} (+/- {np.std(cv_clf['test_accuracy']):.4f})")
print(f"Test F1:       {np.mean(cv_clf['test_f1']):.4f} (+/- {np.std(cv_clf['test_f1']):.4f})")
print(f"Test AUC:      {np.mean(cv_clf['test_auc']):.4f} (+/- {np.std(cv_clf['test_auc']):.4f})")

## 10.4 Built-in CV Variants: `RidgeCV`, `LassoCV`, `ElasticNetCV`, `LogisticRegressionCV`

Sklearn offers CV-aware versions of the regularized estimators that efficiently scan the `alpha` (or `C`) grid internally. They are typically *much* faster than wrapping a plain estimator in `GridSearchCV`, because they reuse computations across the regularization path.

In [ ]:
from sklearn.linear_model import ElasticNetCV, LassoCV, LogisticRegressionCV, RidgeCV

# Regression: tune alpha internally
ridge_cv = RidgeCV(alphas=np.logspace(-4, 4, 50), cv=5).fit(X_train_rs, y_train_r)
print(f"RidgeCV:      best alpha = {ridge_cv.alpha_:.4f}   test R^2 = {ridge_cv.score(X_test_rs, y_test_r):.4f}")

lasso_cv = LassoCV(alphas=100, cv=5, max_iter=20000, random_state=42).fit(X_train_rs, y_train_r)
print(f"LassoCV:      best alpha = {lasso_cv.alpha_:.6f}   test R^2 = {lasso_cv.score(X_test_rs, y_test_r):.4f}")

enet_cv = ElasticNetCV(l1_ratio=[0.1, 0.3, 0.5, 0.7, 0.9, 1.0], alphas=50, cv=5, max_iter=20000, random_state=42).fit(
    X_train_rs, y_train_r
)
print(
    f"ElasticNetCV: best alpha = {enet_cv.alpha_:.6f}   best l1_ratio = {enet_cv.l1_ratio_:.2f}   "
    f"test R^2 = {enet_cv.score(X_test_rs, y_test_r):.4f}"
)

# Classification: tune C internally. l1_ratios=(0.0,) = pure L2; use_legacy_attributes=False
# makes C_ / l1_ratio_ plain floats (the scikit-learn >= 1.10 behaviour); scoring is given
# explicitly because its default changes in a future release.
logreg_cv = LogisticRegressionCV(
    Cs=np.logspace(-3, 3, 20),
    l1_ratios=(0.0,),
    use_legacy_attributes=False,
    scoring="accuracy",
    solver="lbfgs",
    cv=5,
    max_iter=5000,
    n_jobs=-1,
).fit(X_train_cs, y_train_c)
print(
    f"\nLogisticRegressionCV (L2): best C = {logreg_cv.C_:.4f}   "
    f"test accuracy = {logreg_cv.score(X_test_cs, y_test_c):.4f}"
)

# 11. Hyperparameter Tuning with Cross-Validation

**Hyperparameters** are model settings that we choose before training (unlike parameters which are learned during training).

For regularized linear / logistic regression, the key hyperparameters are:
- **`alpha`** (linear) or **`C`** (logistic; $C = 1/(n\lambda)$, larger = weaker penalty): regularization strength (always log-scale).
- **`l1_ratio`**: L1/L2 mixing in $[0, 1]$ — `0` = L2 (Ridge), `1` = L1 (Lasso), in between = Elastic Net. For `LogisticRegression` this replaces the deprecated `penalty=` argument.
- **`solver`** (logistic): must be compatible with the chosen `l1_ratio` (see the table in Section 9).

**Goal**: find the hyperparameter combination that gives the best cross-validation performance — **on the training set**. The searches below therefore see only `X_train_*`; the test split is used once, at the end, to report an unbiased number. Keep in mind that `best_score_` is the *maximum* over many candidates of a noisy CV estimate, so it is optimistically biased (nested CV is the remedy if you need an honest CV number).

## 11.1 Manual Grid Search

In [ ]:
from sklearn.model_selection import cross_val_score

print("=" * 60)
print("MANUAL GRID SEARCH - Elastic Net Logistic Regression")
print("=" * 60)

C_grid = np.logspace(-2, 2, 6)
l1_grid = [0.1, 0.5, 0.9]

records = []
total = len(C_grid) * len(l1_grid)
k = 0
for C in C_grid:
    for rho in l1_grid:
        k += 1
        pipe = Pipeline(
            [
                ("scaler", StandardScaler()),
                (
                    "logreg",
                    LogisticRegression(
                        C=C,
                        l1_ratio=rho,
                        solver="saga",
                        max_iter=10000,
                    ),
                ),
            ]
        )
        scores = cross_val_score(pipe, X_train_c, y_train_c, cv=5, scoring="accuracy", n_jobs=-1)
        mean_s, std_s = scores.mean(), scores.std()
        records.append({"C": C, "l1_ratio": rho, "mean_acc": mean_s, "std_acc": std_s})
        print(f"[{k:2d}/{total}] C={C:<8.3f} l1_ratio={rho:<5}  acc={mean_s:.4f} (+/- {std_s:.4f})")

manual_df = pd.DataFrame(records).sort_values("mean_acc", ascending=False)
print("\nTOP 5 CONFIGURATIONS")
print(manual_df.head().to_string(index=False))

## 11.2 GridSearchCV (Automated Grid Search)

Sklearn's `GridSearchCV` automates the grid search process with additional features:
- Parallel processing
- Automatic cross-validation
- Detailed results tracking
- Easy access to best parameters and scores

In [ ]:
from sklearn.model_selection import GridSearchCV

# Grid search for the regularized linear regression (ElasticNet on housing data)
pipe_gs_reg = Pipeline([("scaler", StandardScaler()), ("enet", ElasticNet(max_iter=20000))])
grid_reg = GridSearchCV(
    pipe_gs_reg,
    param_grid={
        "enet__alpha": np.logspace(-4, 1, 8),
        "enet__l1_ratio": [0.1, 0.5, 0.9, 1.0],
    },
    cv=5,
    scoring="r2",
    n_jobs=-1,
    return_train_score=True,
    verbose=1,
)
grid_reg.fit(X_train_r, y_train_r)
print("[REGRESSION] Best params:", grid_reg.best_params_)
print(
    f"[REGRESSION] Best CV R^2: {grid_reg.best_score_:.4f}  (selection-biased: max over {len(grid_reg.cv_results_['params'])} candidates)"
)
print(f"[REGRESSION] Test R^2 of the refitted best pipeline: {grid_reg.score(X_test_r, y_test_r):.4f}")

# Grid search for L2 logistic regression (classification)
pipe_gs_clf = Pipeline([("scaler", StandardScaler()), ("logreg", LogisticRegression(solver="lbfgs", max_iter=5000))])
grid_clf = GridSearchCV(
    pipe_gs_clf,
    param_grid={
        "logreg__C": np.logspace(-3, 3, 10),
    },
    cv=5,
    scoring="accuracy",
    n_jobs=-1,
    return_train_score=True,
    verbose=1,
)
grid_clf.fit(X_train_c, y_train_c)
print("\n[CLASSIFICATION] Best params:", grid_clf.best_params_)
print(f"[CLASSIFICATION] Best CV accuracy: {grid_clf.best_score_:.4f}  (selection-biased)")
print(f"[CLASSIFICATION] Test accuracy of the refitted best pipeline: {grid_clf.score(X_test_c, y_test_c):.4f}")

## 11.3 RandomizedSearchCV (Automated Random Search)

`RandomizedSearchCV` is particularly useful when:
- The hyperparameter space is very large
- You have limited computational budget
- You want to explore continuous distributions

For `alpha` / `C`, the natural distribution is **log-uniform**: we want equal coverage of each order of magnitude.

In [ ]:
from scipy.stats import loguniform, uniform
from sklearn.model_selection import RandomizedSearchCV

# Elastic Net logistic regression, random search
pipe_rs = Pipeline(
    [
        ("scaler", StandardScaler()),
        ("logreg", LogisticRegression(solver="saga", max_iter=10000)),
    ]
)

random_search = RandomizedSearchCV(
    estimator=pipe_rs,
    param_distributions={
        "logreg__C": loguniform(1e-3, 1e3),
        "logreg__l1_ratio": uniform(loc=0.0, scale=1.0),  # uniform on [0, 1]
    },
    n_iter=30,
    cv=5,
    scoring="accuracy",
    n_jobs=-1,
    random_state=42,
    return_train_score=True,
    verbose=1,
)
random_search.fit(X_train_c, y_train_c)

print("=" * 60)
print("RANDOMIZEDSEARCHCV RESULTS (Elastic Net logistic regression)")
print("=" * 60)
print("Best params:", random_search.best_params_)
print(f"Best CV accuracy: {random_search.best_score_:.4f}  (selection-biased)")
print(f"Test accuracy of the refitted best pipeline: {random_search.score(X_test_c, y_test_c):.4f}")

print("\n" + "=" * 60)
print("COMPARISON: GRIDSEARCH vs RANDOMIZEDSEARCH")
print("=" * 60)
print(
    f"GridSearchCV (L2 logreg):         best accuracy = {grid_clf.best_score_:.4f}  "
    f"({len(grid_clf.cv_results_['params'])} combos)"
)
print(f"RandomizedSearchCV (enet logreg): best accuracy = {random_search.best_score_:.4f}  (30 combos)")
print("(both are CV scores on the training split, maximized over candidates -> compare them with care)")

# 12. Comprehensive Examples: Complete Pipelines

Let's put everything together into complete, production-ready pipelines that incorporate all best practices we've learned.

## Complete Workflow:
1. Load and explore data
2. Split into train/test sets
3. Build pipeline with preprocessing
4. Hyperparameter tuning with cross-validation
5. Evaluate final model on test set
6. Analyze results

## 12.1 Complete Regression Pipeline (Elastic Net)

In [ ]:
print("=" * 70)
print("COMPLETE REGRESSION PIPELINE (Elastic Net on Housing)")
print("=" * 70)

# Step 1: load data (already loaded)
print(f"\n[STEP 1] Dataset shape: {X_reg.shape}")

# Step 2: train/test split BEFORE preprocessing
X_tr_final, X_te_final, y_tr_final, y_te_final = train_test_split(X_reg, y_reg, test_size=0.2, random_state=42)
print(f"\n[STEP 2] Training: {X_tr_final.shape[0]}   Test: {X_te_final.shape[0]}")

# Step 3: pipeline
pipeline_final_reg = Pipeline(
    [
        ("scaler", StandardScaler()),
        ("enet", ElasticNet(max_iter=20000, random_state=42)),
    ]
)
print("\n[STEP 3] Pipeline: StandardScaler -> ElasticNet")

# Step 4: hyperparameter tuning
grid_reg_final = GridSearchCV(
    pipeline_final_reg,
    param_grid={
        "enet__alpha": np.logspace(-4, 1, 10),
        "enet__l1_ratio": [0.1, 0.3, 0.5, 0.7, 1.0],  # l1_ratio=0 is discouraged for ElasticNet (use Ridge)
    },
    cv=5,
    scoring="r2",
    n_jobs=-1,
)
grid_reg_final.fit(X_tr_final, y_tr_final)
print("\n[STEP 4] Best params:", grid_reg_final.best_params_)
print(f"         Best CV R^2: {grid_reg_final.best_score_:.4f}")

# Step 5: evaluate on held-out test set
best_reg = grid_reg_final.best_estimator_
y_pred_reg = best_reg.predict(X_te_final)
print("\n[STEP 5] FINAL TEST METRICS")
print(f"  R^2:  {r2_score(y_te_final, y_pred_reg):.4f}")
print(f"  RMSE: {np.sqrt(mean_squared_error(y_te_final, y_pred_reg)):.4f}")
print(f"  MAE:  {mean_absolute_error(y_te_final, y_pred_reg):.4f}")

## 12.2 Complete Classification Pipeline (Logistic Regression)

In [ ]:
print("=" * 70)
print("COMPLETE CLASSIFICATION PIPELINE (Logistic Regression on Breast Cancer)")
print("=" * 70)

# Step 1
print(f"\n[STEP 1] Dataset shape: {X_clf.shape}   Classes: {cancer.target_names}")

# Step 2: stratified split
X_tr_c, X_te_c, y_tr_c, y_te_c = train_test_split(X_clf, y_clf, test_size=0.2, random_state=42, stratify=y_clf)
print(f"\n[STEP 2] Training: {X_tr_c.shape[0]}   Test: {X_te_c.shape[0]}")

# Step 3: pipeline
pipeline_final_clf = Pipeline(
    [
        ("scaler", StandardScaler()),
        ("logreg", LogisticRegression(solver="saga", max_iter=10000, random_state=42)),
    ]
)
print("\n[STEP 3] Pipeline: StandardScaler -> LogisticRegression(saga)")

# Step 4: hyperparameter tuning (both penalty, C, and l1_ratio)
grid_clf_final = GridSearchCV(
    pipeline_final_clf,
    param_grid={
        "logreg__C": np.logspace(-3, 3, 10),
        "logreg__l1_ratio": [0.0, 0.3, 0.5, 0.7, 1.0],  # 0=L2, 1=L1, in between=Elastic Net
    },
    cv=5,
    scoring="accuracy",
    n_jobs=-1,
)
grid_clf_final.fit(X_tr_c, y_tr_c)
print("\n[STEP 4] Best params:", grid_clf_final.best_params_)
print(f"         Best CV accuracy: {grid_clf_final.best_score_:.4f}")

# Step 5: evaluate on held-out test set
best_clf = grid_clf_final.best_estimator_
y_pred_clf = best_clf.predict(X_te_c)
y_prob_clf = best_clf.predict_proba(X_te_c)[:, 1]

print("\n[STEP 5] FINAL TEST METRICS")
print(f"  Accuracy:  {accuracy_score(y_te_c, y_pred_clf):.4f}")
print(f"  Precision: {precision_score(y_te_c, y_pred_clf):.4f}")
print(f"  Recall:    {recall_score(y_te_c, y_pred_clf):.4f}")
print(f"  F1-score:  {f1_score(y_te_c, y_pred_clf):.4f}")
print(f"  AUC:       {auc(*roc_curve(y_te_c, y_prob_clf)[:2]):.4f}")

print("\nDETAILED CLASSIFICATION REPORT")
print(classification_report(y_te_c, y_pred_clf, target_names=cancer.target_names))

In [ ]:
# Final confusion matrix
cm_final = confusion_matrix(y_te_c, y_pred_clf)

fig, ax = plt.subplots(figsize=(8, 6))
disp = ConfusionMatrixDisplay(confusion_matrix=cm_final, display_labels=cancer.target_names)
disp.plot(cmap="Greens", ax=ax, values_format="d")
plt.title("Final Logistic Regression Model - Confusion Matrix", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

# 13. Key Takeaways and Best Practices

## Summary

**Strengths:**
- ✅ Simple, fast, and highly interpretable.
- ✅ Convex optimization — closed-form (OLS/Ridge) or a global minimum with no local traps (logistic; finite only if the classes are not perfectly separable).
- ✅ Natural probabilistic interpretation (MLE under Gaussian or Bernoulli noise).
- ✅ Statistical inference is available through `statsmodels` (p-values, CIs, F-tests).
- ✅ Regularized variants handle multicollinearity and high-dimensional problems.
- ✅ Lasso / L1 gives automatic variable selection.
- ✅ Strong baseline for any regression or classification task.

**Weaknesses:**
- ❌ Linearity (or log-linearity) assumption — misses non-linear relationships unless you feature-engineer them.
- ❌ Sensitive to outliers (OLS squared loss; logistic is milder but still affected).
- ❌ Unregularized variants blow up under multicollinearity.
- ❌ Require feature scaling for regularized variants.
- ❌ Classical inference relies on strong assumptions.

## Best Practices

### 1. **ALWAYS Scale for Regularized Models**
```python
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('model', Ridge(alpha=1.0)),         # or LogisticRegression(C=1.0)
])
```

### 2. **Use Pipelines and Cross-Validation Together**
- Prevents data leakage (scaling fit on training folds only).
- Everything works seamlessly with `GridSearchCV` / `cross_validate`.

### 3. **Search `alpha` / `C` on a Log Scale**
- `alpha ∈ {1e-4, ..., 1e4}` or `C ∈ {1e-4, ..., 1e4}` — a linear grid is almost always wrong.
- Know the scaling: `Ridge(alpha)` = $n\lambda$, `Lasso`/`ElasticNet(alpha)` = $\lambda$, `LogisticRegression(C)` = $1/(n\lambda)$.
- For `l1_ratio`, a short uniform grid `[0.1, 0.3, 0.5, 0.7, 0.9, 1.0]` is usually enough.

### 4. **Pick the Right Regularization**
- **Plain (OLS / unpenalized logistic)**: when $n \gg p$ and features are uncorrelated.
- **Ridge (L2)**: when features are correlated but all are relevant.
- **Lasso (L1)**: when you believe the true model is sparse (few features matter).
- **Elastic Net**: when features are correlated *and* you want sparsity.

### 5. **Use Built-in CV Variants for Speed**
- `RidgeCV`, `LassoCV`, `ElasticNetCV`, `LogisticRegressionCV` exploit the solution path — orders of magnitude faster than `GridSearchCV`.

### 6. **Match Solver to Penalty (Logistic)** — the penalty is chosen with `l1_ratio` (the `penalty=` argument is deprecated)
| `l1_ratio` | Penalty | Compatible `solver` |
|---|---|---|
| `0.0` (default) | L2 | `lbfgs` (default), `newton-cg`, `newton-cholesky`, `sag`, `saga`, `liblinear` |
| `1.0` | L1 | `liblinear`, `saga` |
| `0 < l1_ratio < 1` | Elastic Net | `saga` only |
| `C=np.inf` | none | `lbfgs`, `newton-cg`, `newton-cholesky`, `sag`, `saga` |

### 7. **Use `statsmodels` for Inference**
- For OLS: `sm.OLS`, for logistic: `sm.Logit` or `sm.GLM(family=Binomial())`.
- Remember to `sm.add_constant(X)`!
- The formula API (`smf.ols`, `smf.logit`, `smf.glm`) handles categorical variables automatically.
- For regularized fits use `fit_regularized()` — but note the p-values / standard errors are NOT returned (the sampling distribution of penalized estimators is non-standard). Mind the `alpha` scaling (Section 5.6 / 9.5) and pass a vector `alpha` with `0` for the constant.
- Unpenalized logistic regression needs classes that are **not** perfectly separable — otherwise the MLE does not exist (Section 8).

### 8. **Always Look at Residual / Diagnostic Plots**
- **Regression**: residuals vs fitted, Q-Q plot, histogram of residuals.
- **Classification**: ROC curve, precision-recall curve, confusion matrix, calibration plot.
- If you see a pattern in residuals or poor calibration, the model is misspecified.

### 9. **Never Leak Test Data**
- Scale and impute inside a `Pipeline` so CV folds are isolated.
- Tune hyperparameters on CV, then evaluate once on the test set.

### 10. **Choose the Right Metrics**
- **Regression**: RMSE (same units as target), MAE (robust), R² (unitless).
- **Classification**: accuracy is misleading on imbalanced data — prefer F1, AUC, precision-recall.
- Always report the spread across CV folds (mean ± std), not just a single number (note: mean ± std is *not* a confidence interval).

## When to Use Linear / Logistic Regression

✅ **Use these models when:**
- You need a fast, interpretable baseline.
- You want statistical inference (hypothesis tests on coefficients).
- The relationship is approximately linear (or the log-odds are).
- You have more samples than features ($n \gg p$), or you use regularization.
- Training speed and prediction latency matter.

❌ **Avoid these models when:**
- The true relationship is strongly non-linear (consider trees, GBMs, neural nets).
- Residuals show strong heteroskedasticity or autocorrelation (consider GLS, robust regression, time-series models).
- You have heavy-tailed outliers (consider Huber, quantile regression).
- The decision boundary is clearly curved and feature engineering is not an option.

## Common Pitfalls to Avoid

1. **Forgetting to scale features before Ridge/Lasso/Elastic Net / regularized logistic** → penalty becomes unfair.
2. **Searching `alpha` / `C` on a linear grid** → wastes compute.
3. **Fitting the scaler on the full dataset** → data leakage.
4. **Mismatched `l1_ratio` / `solver` combination** in sklearn's `LogisticRegression` → error (e.g. `lbfgs` cannot do L1).
5. **Interpreting Lasso coefficients as "causal"** → feature selection ≠ causal discovery.
6. **Reading p-values from regularized fits** → their sampling distributions are non-standard.
7. **Forgetting to `sm.add_constant(X)` in statsmodels** → silently fits without an intercept!
8. **Using accuracy as the only metric on imbalanced data** → model might “predict the majority class” and look great.
9. **Ignoring multicollinearity in OLS / unpenalized logistic** → coefficients become unstable; use VIF diagnostics or switch to Ridge.
10. **Not checking residuals / calibration** → high R² or accuracy can still hide systematic model misspecification.
11. **Fitting unpenalized logistic regression on (quasi-)separable data** → exploding coefficients, NaN p-values, pseudo-R² ≈ 1; regularize or drop features.
12. **Mixing up penalty scalings** (`Ridge` vs `Lasso` `alpha`, sklearn `C` vs statsmodels `alpha`) → a fit that is $n$ times more (or less) regularized than you think.

## Further Reading

- [Scikit-learn Linear Models](https://scikit-learn.org/stable/modules/linear_model.html)
- [Scikit-learn Logistic Regression](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html)
- [Statsmodels Regression Overview](https://www.statsmodels.org/stable/regression.html)
- [Statsmodels GLM (incl. logistic)](https://www.statsmodels.org/stable/glm.html)
- [Elements of Statistical Learning — Chapters 3–4](https://hastie.su.domains/ElemStatLearn/)
- [Zou & Hastie (2005) — Regularization and variable selection via the elastic net](https://web.stanford.edu/~hastie/Papers/B67.2%20(2005)%20301-320%20Zou%20&%20Hastie.pdf)
- [Cross-Validation Techniques](https://scikit-learn.org/stable/modules/cross_validation.html)


# 14. Homework Assignment

You will build and compare regularized linear and logistic regression models on two datasets.

---

## Part A — Regression: Diabetes Progression

### Dataset
Use the **Diabetes dataset** from sklearn:

```python
from sklearn.datasets import load_diabetes

diabetes = load_diabetes()
X, y = diabetes.data, diabetes.target  # X: (442, 10), y: (442,)
feature_names = diabetes.feature_names
```

- 442 patients, 10 baseline features (age, BMI, blood pressure, 6 blood serum measurements, ...)
- Target: a quantitative measure of disease progression one year after baseline
- Features are already centered and scaled to unit norm by sklearn.

### Tasks

#### A1: Data Exploration and Preprocessing
1. Load the Diabetes dataset.
2. Explore: shape, feature names, target statistics, correlation matrix heatmap, feature-vs-target scatterplots.
3. Split into train (70%) / test (30%) with `random_state=42`.

#### A2: From-Scratch and Sklearn
1. Implement closed-form OLS with `np.linalg.solve` (Section 2.1).
2. Fit `sklearn.linear_model.LinearRegression` and verify the coefficients match.
3. Report R², RMSE, MAE on the held-out test set.

#### A3: Statsmodels Inference
1. Fit `statsmodels.api.OLS` (remember to `sm.add_constant(X)`!).
2. Interpret the summary:
   - Which features are statistically significant at $\alpha=0.05$?
   - What is the adjusted R²?
   - Comment on the condition number and the Jarque-Bera p-value.

#### A4: Regularization
1. Fit `Ridge`, `Lasso`, `ElasticNet`.
2. Plot the **coefficient path** for each as a function of `alpha` on a log scale ($10^{-3}$ to $10^{1}$).
3. Compare which features Lasso keeps vs. which Ridge keeps.
4. Replicate with `statsmodels.OLS.fit_regularized` and compare coefficients. Mind the objective (Section 5.6): pass `alpha` as a vector with `0` for the constant, use the *same* `alpha` as sklearn for Lasso / Elastic Net but `alpha_sklearn / n` for Ridge, and compare objective values as well as coefficients.

#### A5: Cross-Validation and Hyperparameter Tuning
1. Manual 5-fold CV with `KFold` (Ridge). Report mean ± std R².
2. Use `cross_validate()` with multiple metrics.
3. Use `RidgeCV`, `LassoCV`, `ElasticNetCV` for automatic alpha selection.
4. Use `GridSearchCV` with:
   - `alpha`: `np.logspace(-4, 1, 10)`
   - `l1_ratio`: `[0.1, 0.3, 0.5, 0.7, 0.9]`
5. Use `RandomizedSearchCV` with `loguniform(1e-4, 1e1)` / `uniform(0,1)`, at least 40 iterations.
6. Compare results, runtimes, and visualize the search (heatmap for grid, scatter for random).

#### A6: Final Model and Analysis
1. Refit the best model on the full training set.
2. Evaluate on the held-out test set (R², RMSE, MAE).
3. Produce residual and Q-Q plots.

---

## Part B — Classification: Pima Indians Diabetes (or UCI Wine Quality)

### Dataset
Use one of the following binary classification datasets. Both are plain CSV files; the loading snippets below were checked to work.

**Option 1 — Pima Indians Diabetes** (768 women, 8 clinical features; target `outcome`: has diabetes = 1 / no = 0). The file is no longer hosted by UCI; a widely used mirror is:

```python
url = "https://raw.githubusercontent.com/jbrownlee/Datasets/master/pima-indians-diabetes.data.csv"
cols = ["pregnancies", "glucose", "blood_pressure", "skin_thickness", "insulin", "bmi", "diabetes_pedigree", "age", "outcome"]
pima = pd.read_csv(url, header=None, names=cols)
# Alternative: from sklearn.datasets import fetch_openml; pima = fetch_openml("diabetes", version=1, as_frame=True).frame
```

**Watch out:** in this dataset a value of `0` in `glucose`, `blood_pressure`, `skin_thickness`, `insulin` and `bmi` is physiologically impossible and encodes a **missing value** — replace those zeros with `NaN` before imputing.

**Option 2 — Wine Quality** (UCI; red wine 1,599 rows, white wine 4,898 rows, 11 physico-chemical features) — binarize as *good* (`quality >= 7`) vs *not good*. This is **not** `sklearn.datasets.load_wine` (a different, 178-row dataset without quality scores):

```python
base = "https://archive.ics.uci.edu/ml/machine-learning-databases/wine-quality/"
red = pd.read_csv(base + "winequality-red.csv", sep=";")  # use red, white, or concatenate both
white = pd.read_csv(base + "winequality-white.csv", sep=";")
red["good"] = (red["quality"] >= 7).astype(int)
```

### Tasks

#### B1: Data Exploration and Preprocessing
1. Load the dataset, explore shape, class distribution, summary statistics, correlation matrix.
2. Check for missing values. Impute or drop as appropriate.
3. Split train (70%) / test (30%) with stratification.

#### B2: From-Scratch and Sklearn
1. Implement binary logistic regression from scratch with cross-entropy + gradient descent (Section 6).
2. Fit `sklearn.linear_model.LogisticRegression` (with large `C` for unregularized).
3. Report accuracy, precision, recall, F1, and AUC on the held-out test set.

#### B3: Statsmodels Inference
1. Fit `sm.Logit(y, add_constant(X))` (or `smf.logit` with a formula).
2. Interpret the summary:
   - Which features are statistically significant?
   - Report the **odds ratio** ($e^{\hat{w}_j}$) and 95% CI for each feature. Is a 10% increase in `X_j` clinically meaningful?
   - Report McFadden pseudo-R² and the LLR p-value.

#### B4: Regularization
1. Fit L2, L1, and Elastic Net logistic regressions (pay attention to solver compatibility).
2. Plot coefficient paths for L1 and L2 as a function of `C` on a log scale.
3. How many features does L1 keep at the best `C`?
4. Replicate in statsmodels and compare (Section 9.5): the L1 fit with `sm.Logit(...).fit_regularized(alpha=1/C, ...)` (L1 only!), the L2 / Elastic Net fits with `sm.GLM(..., family=sm.families.Binomial()).fit_regularized(alpha=1/(C*n), L1_wt=...)`. In both cases pass `alpha` as a vector with `0` for the constant.

#### B5: Cross-Validation and Hyperparameter Tuning
1. Use `StratifiedKFold` (5 folds).
2. Use `LogisticRegressionCV` with `Cs = np.logspace(-3, 3, 20)` for both L1 and L2.
3. Use `GridSearchCV` to jointly tune `C` and `l1_ratio ∈ {0.0, 0.5, 1.0}` (L2 / Elastic Net / L1; use `solver='saga'`).
4. Use `RandomizedSearchCV` with `loguniform(1e-3, 1e3)` for `C` and `uniform(0,1)` for `l1_ratio`.
5. Compare best CV scores, runtimes, and visualize.

#### B6: Final Model and Analysis
1. Refit the best model on the full training set.
2. Evaluate on the held-out test set: accuracy, precision, recall, F1, AUC.
3. Plot ROC curve, precision-recall curve, and confusion matrix.
4. **Bonus**:
   - Which features ended up being selected / shrunk to zero? Does that match domain intuition?
   - Threshold the predicted probability at something other than 0.5 (e.g. 0.3) and observe the precision–recall trade-off.
   - Compare your results with KNN / SVM classification on the same data.
   - If your dataset is imbalanced, re-run with `class_weight='balanced'` and discuss the effect on recall for the minority class.
